In [61]:
import os
import gc
import math
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
import itertools
from sklearn.metrics import roc_auc_score, roc_curve
from model_tools.optimization.anti_overfitting_tuner import AntiOverfittingTuner
from model_tools.optimization.experiment_pipeline import run_experiment_batch
from sklearn.linear_model import LogisticRegression
from model_tools.optimization.objectives import KSObjective,AUCObjective
from model_tools.evaluation.delivery_report import ModelDeliveryReport

# 分数写入
from model_tools.utils import whos
import model_tools as mt
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

# 读取标签与实验配置

本 notebook 只读取输入文件，不回写或修改原始数据。Weight 参数仅使用 `1-train` 训练、`2-ensemble` 非中银验证；`3-oos` 只用于最终评估。

In [38]:
ID_COL = 'test_id'
LABEL_COL = 'flag'
BASE_COL = 'ym_proba_a077_3'
SAMPLE_COL = 'mid_sample_type'
PARTNER_COL = 'partner_name'
PARTNER_CODE_COL = 'partner_code'
ZHONGYIN_NAME = '中银'

LABEL_PATH = '/Users/mayongzhi/Job/Job_xyd/数据/蜜蜂分/2-建模样本_v13-update-标记zy为ensemble样本类型.csv'
HIDDEN_PATH = '/Users/mayongzhi/Job/Job_xyd/数据/数据源/ym/ym_hidden80.pkl'
V4_PATH = '/Users/mayongzhi/Job/Job_xyd/数据/数据源/ym/ym_v4.pkl'

use_cols = [
    ID_COL, LABEL_COL, PARTNER_COL, PARTNER_CODE_COL, SAMPLE_COL, BASE_COL
]
label_df = pd.read_csv(LABEL_PATH, usecols=use_cols)
label_df.columns

Index(['test_id', 'partner_code', 'flag', 'partner_name', 'mid_sample_type',
       'ym_proba_a077_3'],
      dtype='object')

In [3]:
label_df.groupby(SAMPLE_COL)[PARTNER_COL].value_counts(dropna=False)

mid_sample_type      partner_name          
1-train              海尔(借呗)                    496269
                     海尔(字节星选)                  143744
                     海尔(借呗拒绝其他通过)              101217
                     lexin_type5                75861
                     京东借条                       60000
                     浦发银行（三方导流）                 60000
                     天津银行                       55000
                     数禾科技                       49471
                     海尔(字节智选)                   16873
                     海尔(字节星选拒绝其他通过)              5329
2-ensemble           中银                        360000
                     海尔(自营贷中)                  274100
                     海尔(蚂蚁)                    141757
                     weipingfubang_lowerirr     86380
3-oos                海尔(借呗贷中)                  453769
                     weipingfubang_highirr     308859
                     海尔(京东贷中)                  250571
                     海尔(字节星选贷中)       

# 加载并合并 hidden80 + V4 特征

合并前强制检查 `test_id` 非空且唯一。特征表中的标签/渠道等元数据显式排除，只保留数值特征；两个特征源若存在同名特征则直接报错，避免产生 `_x/_y`。

In [39]:
def validate_unique_key(df, df_name, key=ID_COL):
    if not isinstance(df, pd.DataFrame):
        raise TypeError('{} 必须是 pandas DataFrame，实际为 {}'.format(df_name, type(df)))
    if not df.columns.is_unique:
        duplicated = df.columns[df.columns.duplicated()].tolist()
        raise ValueError('{} 存在重复列名: {}'.format(df_name, duplicated[:20]))
    if key not in df.columns:
        raise KeyError('{} 缺少主键 {}'.format(df_name, key))
    if df[key].isna().any():
        raise ValueError('{} 的 {} 存在空值'.format(df_name, key))
    duplicated_count = int(df[key].duplicated().sum())
    if duplicated_count:
        raise ValueError('{} 的 {} 存在 {} 个重复值'.format(df_name, key, duplicated_count))


def validate_binary_values(series, series_name, require_both=False):
    values = set(pd.Series(series).dropna().unique().tolist())
    if not values.issubset({0, 1, 0.0, 1.0}):
        raise ValueError('{} 只能包含 0/1，实际发现 {}'.format(series_name, sorted(values)))
    if require_both and values != {0, 1}:
        raise ValueError('{} 必须同时包含好坏两类，实际为 {}'.format(series_name, sorted(values)))


validate_unique_key(label_df, 'label_df')
validate_binary_values(label_df[LABEL_COL], 'label_df.flag')

# label_df 中的字段以标签文件为准，特征源内同名元数据不参与建模。
label_row_count = len(label_df)
forbidden_feature_cols = set(label_df.columns) - {ID_COL}
# forbidden_feature_cols 添加'test_id', 'id_number', 'mobile', 'apply_date', 'code','hxs1'
forbidden_feature_cols.update({'test_id', 'id_number', 'mobile', 'apply_date', 'code','hxs1'})

# 两个特征文件很大，顺序加载、裁列、合并并释放，避免同时驻留内存。
hidden_raw = joblib.load(HIDDEN_PATH)
hidden_raw.drop_duplicates(ID_COL, keep='first', inplace=True)
validate_unique_key(hidden_raw, 'ym_hidden80')
hidden_row_count = len(hidden_raw)
hidden_feature_cols = [
    c for c in hidden_raw.columns
    if c != ID_COL and c not in forbidden_feature_cols and pd.api.types.is_numeric_dtype(hidden_raw[c])
]
if not hidden_feature_cols:
    raise ValueError('ym_hidden80 必须至少包含一个可用数值特征')
hidden_drop_cols = [c for c in hidden_raw.columns if c != ID_COL and c not in hidden_feature_cols]
hidden_raw.drop(columns=hidden_drop_cols, inplace=True)
model_df = label_df.merge(hidden_raw, on=ID_COL, how='inner', validate='one_to_one')
del hidden_raw, hidden_drop_cols
gc.collect()

v4_raw = joblib.load(V4_PATH)
validate_unique_key(v4_raw, 'ym_v4')
v4_row_count = len(v4_raw)
v4_feature_cols = [
    c for c in v4_raw.columns
    if c != ID_COL and c not in forbidden_feature_cols and pd.api.types.is_numeric_dtype(v4_raw[c])
]
if not v4_feature_cols:
    raise ValueError('ym_v4 必须至少包含一个可用数值特征')
overlap_features = sorted(set(hidden_feature_cols) & set(v4_feature_cols))
if overlap_features:
    raise ValueError('hidden80 与 V4 存在同名数值特征，请显式重命名后再运行: {}'.format(overlap_features[:30]))
v4_drop_cols = [c for c in v4_raw.columns if c != ID_COL and c not in v4_feature_cols]
v4_raw.drop(columns=v4_drop_cols, inplace=True)
model_df = model_df.merge(v4_raw, on=ID_COL, how='inner', validate='one_to_one')
del v4_raw, v4_drop_cols
gc.collect()

feature_cols = hidden_feature_cols + v4_feature_cols
validate_unique_key(model_df, 'model_df')
merge_audit = pd.DataFrame({
    'dataset': ['label_df', 'ym_hidden80', 'ym_v4', 'merged_model_df'],
    'row_count': [label_row_count, hidden_row_count, v4_row_count, len(model_df)],
    'feature_count': [0, len(hidden_feature_cols), len(v4_feature_cols), len(feature_cols)]
})
display(merge_audit)

,dataset,row_count,feature_count
0,label_df,6821455,0
1,ym_hidden80,4640713,80
2,ym_v4,4506552,262
3,merged_model_df,4353504,342


In [40]:
model_df.head()

,test_id,partner_code,flag,partner_name,mid_sample_type,ym_proba_a077_3,hxe1,hxe2,hxe3,hxe4,hxe5,hxe6,hxe7,hxe8,hxe9,hxe10,hxe11,hxe12,hxe13,hxe14,hxe15,hxe16,hxe17,hxe18,hxe19,hxe20,hxe21,hxe22,hxe23,hxe24,hxe25,hxe26,hxe27,hxe28,hxe29,hxe30,hxe31,hxe32,hxe33,hxe34,hxe35,hxe36,hxe37,hxe38,hxe39,hxe40,hxe41,hxe42,hxe43,hxe44,hxe45,hxe46,hxe47,hxe48,hxe49,hxe50,hxe51,hxe52,hxe53,hxe54,hxe55,hxe56,hxe57,hxe58,hxe59,hxe60,hxe61,hxe62,hxe63,hxe64,hxe65,hxe66,hxe67,hxe68,hxe69,hxe70,hxe71,hxe72,hxe73,hxe74,hxe75,hxe76,hxe77,hxe78,hxe79,hxe80,hce1,hce10,hce11,hce12,hce13,hce14,hce15,hce16,hce17,hce18,hce19,hce2,hce20,hce21,...,hcu119,hcu12,hcu120,hcu121,hcu122,hcu13,hcu14,hcu15,hcu16,hcu17,hcu18,hcu19,hcu2,hcu20,hcu21,hcu22,hcu23,hcu24,hcu25,hcu26,hcu27,hcu28,hcu29,hcu3,hcu30,hcu31,hcu32,hcu33,hcu34,hcu35,hcu36,hcu37,hcu38,hcu39,hcu4,hcu40,hcu41,hcu42,hcu43,hcu44,hcu45,hcu46,hcu47,hcu48,hcu49,hcu5,hcu50,hcu51,hcu52,hcu53,hcu54,hcu55,hcu56,hcu57,hcu58,hcu59,hcu6,hcu60,hcu61,hcu62,hcu63,hcu64,hcu65,hcu66,hcu67,hcu68,hcu69,hcu7,hcu70,hcu71,hcu72,hcu73,hcu74,hcu75,hcu76,hcu77,hcu78,hcu79,hcu8,hcu80,hcu81,hcu82,hcu83,hcu84,hcu85,hcu86,hcu87,hcu88,hcu89,hcu9,hcu90,hcu91,hcu92,hcu93,hcu94,hcu95,hcu96,hcu97,hcu98,hcu99
0,1184164,c_4,1.0,恒昌_new,9-不参与标品建模,0.047669,0.000000,0.0,0.294604,0.191469,0.000000,0.0,0.0,0.0,0.328571,0.0,0.149731,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.108740,0.579989,0.0,0.105191,0.0,0.0,0.0,0.0,0.274775,0.0,0.000000,0.000000,0.0,0.0,0.0,0.217793,0.0,0.0,0.0,0.000000,0.0,0.213713,0.0,0.0,0.0,0.0,0.096760,0.000000,0.375428,0.000000,0.343175,0.0,0.057567,0.239317,0.000000,0.000000,0.558383,0.0,0.0,0.122246,0.0,0.0,0.260008,0.0,0.0,0.000000,0.0,0.0,0.300628,0.275943,0.239412,0.670533,0.0,0.469973,0.0,0.401437,0.0,0.644622,0.0,0.0,0.0,0.0,24.0,1.0,1.0,1.0,25.0,1.0,1.0,1.0,25.0,5.0,5.0,1.0,24.0,14.0,...,0.6759,0.0081,0.9260,1.3289,0.0,0.0185,0.3553,0.1590,0.9260,0.0364,0.0282,0.0030,0.0096,0.0307,0.0159,0.0052,0.0022,0.0015,0.5756,0.0850,0.1250,0.2143,0.2071,0.1625,0.4558,0.3153,0.2289,0.4738,0.0848,0.2414,0.3289,0.2464,0.1833,0.3838,0.5139,0.2629,0.1404,0.2299,0.4786,0.0075,0.3851,0.0094,0.0053,0.0269,0.6280,0.0352,0.3720,0.7947,0.2053,0.2781,0.7219,0.9260,0.0740,0.6222,0.3778,0.8674,0.0699,0.1326,0.6720,0.3280,0.5873,0.4127,0.4777,-4.0279,-2.6505,0.1814,1.3325,0.0340,0.5211,-0.0077,-0.7270,-3.9033,0.1723,-0.4372,-0.3513,0.9062,0.0130,-0.0204,0.0014,-2.8365,2.2087,0.8619,-0.5056,-1.1674,-2.2847,-3.5010,1.1654,-0.7472,0.5398,0.1059,1.1714,0.4826,1.3773,-0.3433,0.1001,-0.1753,1.2343,0.8561,0.6448,0.2291
1,1184172,c_4,1.0,恒昌_new,9-不参与标品建模,0.045002,0.000000,0.0,0.000000,0.160462,0.000000,0.0,0.0,0.0,0.231178,0.0,0.120275,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.075306,0.503412,0.0,0.096205,0.0,0.0,0.0,0.0,0.234836,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.194873,0.0,0.0,0.0,0.0,0.073831,0.000000,0.341765,0.000000,0.044217,0.0,0.057769,0.182490,0.000000,0.000000,0.479781,0.0,0.0,0.097749,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.001899,0.209324,0.193009,0.584803,0.0,0.441566,0.0,0.352031,0.0,0.435723,0.0,0.0,0.0,0.0,23.0,1.0,8.0,1.0,23.0,1.0,6.0,5.0,22.0,6.0,7.0,1.0,21.0,17.0,...,0.4033,0.0307,0.9112,0.3719,0.0,0.0654,0.2383,0.0575,0.9112,0.0469,0.0246,0.0002,0.0611,0.0031,0.0012,0.0003,0.0002,0.0001,0.4225,0.1628,0.1955,0.2193,0.4282,0.4064,0.4738,0.2553,0.2709,0.4099,0.1220,0.3719,0.2990,0.2134,0.1157,0.3770,0.1382,0.2584,0.1525,0.6471,0.1423,0.0673,0.0680,0.5320,0.0427,0.0120,0.6871,0.1033,0.3129,0.5589,0.4411,0.2526,0.7474,0.9147,0.0853,0.5588,0.4412,0.7612,0.0909,0.2388,0.7406,0.2594,0.2347,0.7653,0.4181,-2.9572,-0.7962,1.0985,0.0197,0.0100,-1.8742,0.3160,-1.8873,-5.1495,0.3070,0.4575,-0.8306,1.0344,-1.0456,-0.8753,0.0004,-4.4882,3.2388,-0.5575,-1.9478,-2.9350,-4.4199,-6.0786,0.7419,-0.2117,0.2174,0.1175,1.2933,0.7345,0.9330,-0.2790,0.5158,-0.6522,0.9404,0.5628,0.3653,0.0356
2,1184175,c_4,1.0,恒昌_new,9-不参与标品建模,0.031715,0.223170,0.0,0.575445,0.348989,0.128978,0.0,0.0,0.0,0.407635,0.0,0.221861,0.0,0.0,0.0,0.0,0.0,0.

# 固定模型参数、指标与 weight 实现

In [ ]:
gp_col = 'partner_name'
target = 'flag'
model_df.groupby([gp_col]).agg({target:[len,sum,'mean']})

flag                   
                           len      sum      mean
partner_name                                     
hc_A1_mob3               20197   1607.0  0.079566
hc_A1_mob6                7675    488.0  0.063583
hc_A2_mob3               62768   9425.0  0.150156
hc_A2_mob6               41829   6790.0  0.162328
hc_A3_mob3               36533   4275.0  0.117017
hc_A3_mob6               27767   3518.0  0.126697
lexin_type1             213198   7282.0  0.034156
lexin_type2              59868   1798.0  0.030033
lexin_type3             112105   4357.0  0.038865
lexin_type4              66013   3628.0  0.054959
lexin_type5              34036   1247.0  0.036638
tongchengjr               1815    262.0  0.144353
weipingfubang_highirr   270176   7673.0  0.028400
weipingfubang_lowerirr   74613   1805.0  0.024191
中银                      360000  35000.0  0.097222
众安小贷复贷常规                152616   4816.0  0.031556
众安小贷复贷自然                 10228    409.0  0.039988
众安小贷新贷常规                 50659   2585.0  0.051027
众安小贷新贷自然                 16863    862.0  0.051118
华通银行-星选                      1      0.0  0.000000
恒昌_new                    5265    448.0  0.085090
恒昌_old                   12017    716.0  0.059582
恒昌客群A                    47323   2357.0  0.049807
恒昌客群B                    21104   1222.0  0.057904
恒昌客群C                    65902   3757.0  0.057009
洋钱罐                      52585   2692.0  0.051193
海尔(京东贷中)                236044   5089.0  0.021560
海尔(借呗)                  452145   9058.0  0.020033
海尔(借呗拒绝其他通过)             95095   3854.0  0.040528
海尔(借呗贷中)                421934  57342.0  0.135903
海尔(字节星选)                133260   3891.0  0.029199
海尔(字节星选拒绝其他通过)            5034    199.0  0.039531
海尔(字节星选贷中)              219877   3700.0  0.016828
海尔(字节智选)                 15447    506.0  0.032757
海尔(字节智选贷中)              136522   1703.0  0.012474
海尔(自营贷中)                255730  30311.0  0.118527
海尔(蚂蚁)                  130285  11223.0  0.086142
海尔(蚂蚁API贷中)             215576  12695.0  0.058889
维新复贷                    122476  50189.0  0.409786
维新复贷基测                   35474   6348.0  0.178948
维新新贷                     43186  16189.0  0.374867
维新新贷基测                   12263   1014.0  0.082688

In [42]:
model_df[gp_col].value_counts()[model_df[gp_col].value_counts() < 3000].index

Index([], dtype='object', name='partner_name')

In [41]:
model_df = model_df[model_df[gp_col].isin(model_df[gp_col].value_counts()[model_df[gp_col].value_counts() > 3000].index)]

In [43]:
model_df = model_df[model_df['mid_sample_type'] != '9-不参与标品建模']

In [44]:
model_df['weight_group'] = (
    model_df[SAMPLE_COL].astype('string') + '_' + model_df[PARTNER_CODE_COL].astype('string')
)
model_df['weight_group'].value_counts()

weight_group
1-train_b_11       452145
3-oos_b_15         421934
2-ensemble_b_22    360000
3-oos_b_6          270176
2-ensemble_b_18    255730
3-oos_b_19         236044
3-oos_b_16         219877
3-oos_b_20         215576
3-oos_b_17         136522
1-train_b_14       133260
2-ensemble_b_9     130285
1-train_b_10        95095
2-ensemble_b_7      74613
3-oos_c_21          66013
3-oos_c_19          59868
1-train_c_22        34036
1-train_b_12        15447
1-train_b_13         5034
Name: count, dtype: Int64

In [ ]:
BASE_CNT_GRID = [30000, 60000, 70000, 90000]
BASE_RATE_GRID = [0.04, 0.07, 0.10, 0.13]
AUC_TOLERANCE = 1e-4

# 与参考 notebook params1 数值等价，改用 sklearn API 的规范参数名。
LGB_PARAMS = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.046,
    'max_depth': 4,
    'n_estimators': 150,
    'num_leaves': 31,
    'min_child_weight': 25,
    'colsample_bytree': 0.6,
    'subsample': 0.8,
    'random_state': 2078,
    'reg_lambda': 2,
    'reg_alpha': 2,
    'n_jobs': 12,
    'bagging_freq': 11,
    'max_cat_threshold': 12,
    'max_bin': 183,
    'verbosity': -1
}

model_df['weight_group'] = (
    model_df[SAMPLE_COL].astype('string') + '_' + model_df[PARTNER_CODE_COL].astype('string')
)


def balance_df_weight(df, gp_col, label_col, base_cnt, base_rate):
    if base_cnt <= 0:
        raise ValueError('base_cnt 必须大于 0')
    if not 0 < base_rate < 1:
        raise ValueError('base_rate 必须位于 (0, 1)')
    if df.empty:
        raise ValueError('计算 weight 的数据不能为空')
    if df[gp_col].isna().any():
        raise ValueError('{} 存在空值，无法按组计算 weight'.format(gp_col))
    if df[label_col].isna().any():
        raise ValueError('{} 存在空值，无法计算 weight'.format(label_col))
    validate_binary_values(df[label_col], label_col, require_both=True)

    weights = pd.Series(np.nan, index=df.index, dtype='float64', name='sample_weight')
    invalid_groups = []
    for group_name, group_df in df.groupby(gp_col, sort=False, dropna=False):
        n_pos = int((group_df[label_col] == 1).sum())
        n_neg = int((group_df[label_col] == 0).sum())
        if n_pos == 0 or n_neg == 0:
            invalid_groups.append((group_name, n_pos, n_neg))
            continue
        weights.loc[group_df.index[group_df[label_col] == 1]] = base_cnt * base_rate / n_pos
        weights.loc[group_df.index[group_df[label_col] == 0]] = base_cnt * (1 - base_rate) / n_neg

    if invalid_groups:
        raise ValueError('以下 weight_group 缺少一个标签类别: {}'.format(invalid_groups[:20]))
    if weights.isna().any() or not np.isfinite(weights.to_numpy()).all() or (weights <= 0).any():
        raise ValueError('生成的 sample_weight 含 NaN/inf/非正数')
    return weights


def binary_score_metrics(y_true, score):
    y = pd.Series(y_true).reset_index(drop=True)
    s = pd.Series(score).reset_index(drop=True)
    valid = y.notna() & s.notna() & np.isfinite(s)
    y = y.loc[valid]
    s = s.loc[valid]
    if len(y) == 0 or y.nunique() < 2:
        return {'auc': np.nan, 'ks': np.nan, 'score_direction': '不可判断'}
    validate_binary_values(y, 'metric_label', require_both=True)
    auc = float(roc_auc_score(y, s))
    fpr, tpr, _ = roc_curve(y, s)
    ks = float(np.max(np.abs(tpr - fpr)))
    direction = 'higher_is_bad' if auc >= 0.5 else 'lower_is_bad'
    return {'auc': auc, 'ks': ks, 'score_direction': direction}


def fit_lgbm(train_df, sample_weight=None):
    if train_df.empty:
        raise ValueError('训练集不能为空')
    if train_df[LABEL_COL].isna().any():
        raise ValueError('训练标签存在空值')
    validate_binary_values(train_df[LABEL_COL], 'train.flag', require_both=True)
    model = lgb.LGBMClassifier(**LGB_PARAMS)
    model.fit(
        train_df[feature_cols],
        train_df[LABEL_COL].astype(int),
        sample_weight=sample_weight
    )
    return model

# Weight 参数扫描

只在 `1-train` 拟合候选模型，并在 `2-ensemble` 和 `3-oos`样本上验证。先取与最佳 AUC 相差不超过 `AUC_TOLERANCE` 的候选，再按 KS、AUC 排序选优；

In [ ]:
weight_train_mask = (model_df[SAMPLE_COL] == '1-train') & model_df[LABEL_COL].notna()
weight_valid_mask = (
    (model_df[SAMPLE_COL] == '2-ensemble')
    & (model_df[PARTNER_COL] != ZHONGYIN_NAME)
    & model_df[LABEL_COL].notna()
)
if (model_df.loc[weight_train_mask, SAMPLE_COL] == '3-oos').any() or (model_df.loc[weight_valid_mask, SAMPLE_COL] == '3-oos').any():
    raise AssertionError('禁止使用 3-oos 进行 weight 参数选优')

weight_train_df = model_df.loc[weight_train_mask]
weight_valid_df = model_df.loc[weight_valid_mask]
validate_binary_values(weight_train_df[LABEL_COL], 'weight_train.flag', require_both=True)
validate_binary_values(weight_valid_df[LABEL_COL], 'weight_valid.flag', require_both=True)

weight_search_rows = []
baseline_model = fit_lgbm(weight_train_df)
baseline_valid_pred = baseline_model.predict_proba(weight_valid_df[feature_cols])[:, 1]
baseline_metrics = binary_score_metrics(weight_valid_df[LABEL_COL], baseline_valid_pred)
weight_search_rows.append({
    'candidate_type': 'unweighted_baseline',
    'base_cnt': np.nan,
    'base_rate': np.nan,
    'val_auc': baseline_metrics['auc'],
    'val_ks': baseline_metrics['ks'],
    'score_direction': baseline_metrics['score_direction']
})
del baseline_model, baseline_valid_pred
gc.collect()

for base_cnt, base_rate in itertools.product(BASE_CNT_GRID, BASE_RATE_GRID):
    candidate_weight = balance_df_weight(
        weight_train_df, 'weight_group', LABEL_COL, base_cnt, base_rate
    )
    candidate_model = fit_lgbm(weight_train_df, sample_weight=candidate_weight)
    valid_pred = candidate_model.predict_proba(weight_valid_df[feature_cols])[:, 1]
    metrics = binary_score_metrics(weight_valid_df[LABEL_COL], valid_pred)
    weight_search_rows.append({
        'candidate_type': 'weighted',
        'base_cnt': int(base_cnt),
        'base_rate': float(base_rate),
        'val_auc': metrics['auc'],
        'val_ks': metrics['ks'],
        'score_direction': metrics['score_direction']
    })
    del candidate_weight, candidate_model, valid_pred
    gc.collect()

weight_search_result = pd.DataFrame(weight_search_rows)
weighted_candidates = weight_search_result.loc[
    weight_search_result['candidate_type'].eq('weighted')
].dropna(subset=['val_auc', 'val_ks']).copy()
if weighted_candidates.empty:
    raise ValueError('没有可用于选优的 weight 候选结果')
max_weight_auc = weighted_candidates['val_auc'].max()
auc_near_best = weighted_candidates.loc[
    weighted_candidates['val_auc'] >= max_weight_auc - AUC_TOLERANCE
].sort_values(['val_ks', 'val_auc'], ascending=[False, False])
best_weight_row = auc_near_best.iloc[0]
best_weight_params = {
    'base_cnt': int(best_weight_row['base_cnt']),
    'base_rate': float(best_weight_row['base_rate']),
    'selection_max_auc': float(max_weight_auc),
    'selected_val_auc': float(best_weight_row['val_auc']),
    'selected_val_ks': float(best_weight_row['val_ks']),
    'auc_tolerance': AUC_TOLERANCE
}
display(weight_search_result.sort_values(['candidate_type', 'val_auc', 'val_ks'], ascending=[True, False, False]))
display(best_weight_params)

# 参数选优完成后释放大表切片，避免与后续八组训练同时占用内存。
del weight_train_df, weight_valid_df, weighted_candidates, auc_near_best
gc.collect()

# 八组固定 LightGBM 实验

四种训练范围为：S1=`1-train`；S2=`1-train + 2-ensemble中银`；S3=`1-train + 2-ensemble非中银`；S4=`1-train + 全部2-ensemble`。每种范围分别训练 unweighted/weighted 模型，共八组。所有模型共用同一组特征和 LightGBM 参数。

In [35]:
BASE_COL = 'ym_proba_a077_3'
prediction_columns = [ID_COL, LABEL_COL, PARTNER_COL, PARTNER_CODE_COL, SAMPLE_COL, BASE_COL]


In [45]:
best_weight_params

{'base_cnt': 30000,
 'base_rate': 0.04,
 'selection_max_auc': 0.6405420966534777,
 'selected_val_auc': 0.6405420966534777,
 'selected_val_ks': 0.203264889220012,
 'auc_tolerance': 0.0001}

In [46]:
labeled_mask = model_df[LABEL_COL].notna()
train_1_mask = model_df[SAMPLE_COL].eq('1-train')
ensemble_mask = model_df[SAMPLE_COL].eq('2-ensemble')
zhongyin_mask = model_df[PARTNER_COL].eq(ZHONGYIN_NAME)
oos_mask = model_df[SAMPLE_COL].eq('3-oos') & labeled_mask

TRAIN_SCENARIOS = {
    'S1_train_only': train_1_mask & labeled_mask,
    'S2_train_plus_ensemble_zhongyin': (train_1_mask | (ensemble_mask & zhongyin_mask)) & labeled_mask,
    'S3_train_plus_ensemble_non_zhongyin': (train_1_mask | (ensemble_mask & ~zhongyin_mask)) & labeled_mask,
    'S4_train_plus_all_ensemble': (train_1_mask | ensemble_mask) & labeled_mask
}

if not oos_mask.any():
    raise ValueError('合并特征后没有可用的 3-oos 有标签样本')
validate_binary_values(model_df.loc[oos_mask, LABEL_COL], '3-oos.flag', require_both=True)

prediction_columns = [ID_COL, LABEL_COL, PARTNER_COL, PARTNER_CODE_COL, SAMPLE_COL, BASE_COL]
prediction_df = model_df.loc[labeled_mask, prediction_columns].copy()
experiment_rows = []
experiment_order = []

for scenario_name, scenario_mask in TRAIN_SCENARIOS.items():
    scenario_df = model_df.loc[scenario_mask]
    if scenario_df[SAMPLE_COL].eq('3-oos').any():
        raise AssertionError('{} 训练范围意外包含 3-oos'.format(scenario_name))
    validate_binary_values(scenario_df[LABEL_COL], '{}.flag'.format(scenario_name), require_both=True)

    for weighted in (False, True):
        weight_name = 'weighted' if weighted else 'unweighted'
        experiment_name = '{}_{}'.format(scenario_name, weight_name)
        experiment_order.append(experiment_name)
        sample_weight = None
        if weighted:
            sample_weight = balance_df_weight(
                scenario_df,
                'weight_group',
                LABEL_COL,
                best_weight_params['base_cnt'],
                best_weight_params['base_rate']
            )

        experiment_model = fit_lgbm(scenario_df, sample_weight=sample_weight)
        all_pred = experiment_model.predict_proba(model_df.loc[labeled_mask, feature_cols])[:, 1]
        if not np.isfinite(all_pred).all():
            raise ValueError('{} 的全样本预测包含 NaN/inf'.format(experiment_name))
        prediction_df[experiment_name] = all_pred
        experiment_rows.append({
            'experiment': experiment_name,
            'scenario': scenario_name,
            'weighted': weighted,
            'train_n': int(len(scenario_df)),
            'train_bad_count': int(scenario_df[LABEL_COL].sum()),
            'train_bad_rate': float(scenario_df[LABEL_COL].mean()),
            'base_cnt': best_weight_params['base_cnt'] if weighted else np.nan,
            'base_rate': best_weight_params['base_rate'] if weighted else np.nan
        })
        del experiment_model, all_pred, sample_weight
        gc.collect()

experiment_registry = pd.DataFrame(experiment_rows)
if len(experiment_registry) != 8 or experiment_registry['experiment'].nunique() != 8:
    raise AssertionError('实验数量必须严格等于 8')
oos_prediction_df = prediction_df.loc[prediction_df[SAMPLE_COL].eq('3-oos')].copy()
display(experiment_registry)

,experiment,scenario,weighted,train_n,train_bad_count,train_bad_rate,base_cnt,base_rate
0,S1_train_only_unweighted,S1_train_only,False,735017,18755,0.025516,NaN,NaN
1,S1_train_only_weighted,S1_train_only,True,735017,18755,0.025516,30000.0,0.04
2,S2_train_plus_ensemble_zhongyin_unweighted,S2_train_plus_ensemble_zhongyin,False,1095017,53755,0.049091,NaN,NaN
3,S2_train_plus_ensemble_zhongyin_weighted,S2_train_plus_ensemble_zhongyin,True,1095017,53755,0.049091,30000.0,0.04
4,S3_train_plus_ensemble_non_zhongyin_unweighted,S3_train_plus_ensemble_non_zhongyin,False,1195645,62094,0.051933,NaN,NaN
5,S3_train_plus_ensemble_non_zhongyin_weighted,S3_train_plus_ensemble_non_zhongyin,True,1195645,62094,0.051933,30000.0,0.04
6,S4_train_plus_all_ensemble_unweighted,S4_train_plus_all_ensemble,False,1555645,97094,0.062414,NaN,NaN
7,S4_train_plus_all_ensemble_weighted,S4_train_plus_all_ensemble,True,1555645,97094,0.062414,30000.0,0.04


# 按 mid_sample_type 与 Base 对比

每个 `mid_sample_type` 分别输出 overall、各 partner_name 和渠道宏平均 mean。mean 是 partner_name 行数值指标的不加权算术平均。每个对比均使用 `flag`、Base 和对应模型预测都非空的共同样本；AUC 使用原始分数，不自动执行 `1-score`。其中仅 `3-oos` 属于正式 OOS 结果，其他样本类型仅用于诊断。

In [47]:
def compare_model_with_base(eval_df, pred_col, sample_type, scope_type, scope_value):
    labeled_count = int(eval_df[LABEL_COL].notna().sum())
    common_mask = (
        eval_df[LABEL_COL].notna()
        & eval_df[BASE_COL].notna()
        & eval_df[pred_col].notna()
        & np.isfinite(eval_df[BASE_COL])
        & np.isfinite(eval_df[pred_col])
    )
    common_df = eval_df.loc[common_mask]
    model_metrics = binary_score_metrics(common_df[LABEL_COL], common_df[pred_col])
    base_metrics = binary_score_metrics(common_df[LABEL_COL], common_df[BASE_COL])
    n = int(len(common_df))
    bad_count = int(common_df[LABEL_COL].sum()) if n else 0
    bad_rate = float(common_df[LABEL_COL].mean()) if n else np.nan
    coverage = n / labeled_count if labeled_count else np.nan
    return {
        SAMPLE_COL: sample_type,
        'evaluation_kind': 'out_of_sample' if sample_type == '3-oos' else 'diagnostic_non_oos',
        'scope_type': scope_type,
        'scope_value': scope_value,
        'experiment': pred_col,
        'partner_count': int(eval_df[PARTNER_COL].nunique(dropna=False)),
        'n': n,
        'bad_count': bad_count,
        'bad_rate': bad_rate,
        'coverage': coverage,
        'model_auc': model_metrics['auc'],
        'model_ks': model_metrics['ks'],
        'model_score_direction': model_metrics['score_direction'],
        'base_auc': base_metrics['auc'],
        'base_ks': base_metrics['ks'],
        'base_score_direction': base_metrics['score_direction'],
        'delta_auc': model_metrics['auc'] - base_metrics['auc'],
        'delta_ks': model_metrics['ks'] - base_metrics['ks']
    }


MEAN_NUMERIC_COLS = [
    'n', 'bad_count', 'bad_rate', 'coverage',
    'model_auc', 'model_ks', 'base_auc', 'base_ks', 'delta_auc', 'delta_ks'
]


def build_partner_macro_mean(partner_rows, sample_type, experiment_name):
    partner_result = pd.DataFrame(partner_rows)
    if partner_result.empty:
        raise ValueError('{} / {} 没有可汇总的 partner_name 结果'.format(sample_type, experiment_name))
    mean_values = partner_result[MEAN_NUMERIC_COLS].mean(skipna=True).to_dict()
    model_directions = partner_result['model_score_direction'].dropna().unique().tolist()
    base_directions = partner_result['base_score_direction'].dropna().unique().tolist()
    mean_row = {
        SAMPLE_COL: sample_type,
        'evaluation_kind': 'out_of_sample' if sample_type == '3-oos' else 'diagnostic_non_oos',
        'scope_type': 'mean',
        'scope_value': 'partner_macro_mean',
        'experiment': experiment_name,
        'partner_count': int(len(partner_result)),
        'model_score_direction': model_directions[0] if len(model_directions) == 1 else 'mixed',
        'base_score_direction': base_directions[0] if len(base_directions) == 1 else 'mixed'
    }
    mean_row.update(mean_values)
    return mean_row


result_rows = []
for sample_type, sample_df in prediction_df.groupby(SAMPLE_COL, dropna=False, sort=True):
    sample_type_value = str(sample_type)
    for experiment_name in experiment_order:
        result_rows.append(compare_model_with_base(
            sample_df, experiment_name, sample_type_value, 'overall', sample_type_value
        ))
        partner_rows = []
        for partner_name, partner_df in sample_df.groupby(PARTNER_COL, dropna=False, sort=True):
            partner_row = compare_model_with_base(
                partner_df, experiment_name, sample_type_value, 'partner_name', str(partner_name)
            )
            partner_rows.append(partner_row)
            result_rows.append(partner_row)
        result_rows.append(build_partner_macro_mean(
            partner_rows, sample_type_value, experiment_name
        ))

result_long = pd.DataFrame(result_rows)
scope_order = {'overall': 0, 'mean': 1, 'partner_name': 2}
result_long['_scope_order'] = result_long['scope_type'].map(scope_order)
result_long = result_long.sort_values(
    [SAMPLE_COL, '_scope_order', 'scope_value', 'experiment']
).drop(columns='_scope_order').reset_index(drop=True)

# 截图式宽表：每个 mid_sample_type 下，列为 Base + 八组实验。
metric_mapping = {
    'partner_count': ('partner_count', 'partner_count'),
    'n': ('n', 'n'),
    'bad_count': ('bad_count', 'bad_count'),
    'bad_rate': ('bad_rate', 'bad_rate'),
    'coverage': ('coverage', 'coverage'),
    'auc': ('base_auc', 'model_auc'),
    'ks': ('base_ks', 'model_ks')
}
wide_blocks = []
scope_cols = [SAMPLE_COL, 'evaluation_kind', 'scope_type', 'scope_value']
for metric_name, (base_metric_col, model_metric_col) in metric_mapping.items():
    model_wide = result_long.pivot(
        index=scope_cols, columns='experiment', values=model_metric_col
    ).reindex(columns=experiment_order)
    base_series = result_long.groupby(scope_cols, dropna=False)[base_metric_col].first()
    metric_wide = pd.concat([base_series.rename('Base'), model_wide], axis=1).reset_index()
    metric_wide.insert(len(scope_cols), 'metric', metric_name)
    wide_blocks.append(metric_wide)
result_wide = pd.concat(wide_blocks, ignore_index=True)
result_wide = result_wide[scope_cols + ['metric', 'Base'] + experiment_order]
result_wide['_scope_order'] = result_wide['scope_type'].map(scope_order)
result_wide = result_wide.sort_values(
    [SAMPLE_COL, '_scope_order', 'scope_value', 'metric']
).drop(columns='_scope_order').reset_index(drop=True)

# 同时提供按 mid_sample_type 拆分的结果，便于逐类查看或导出。
result_long_by_sample_type = {
    str(sample_type): sample_result.reset_index(drop=True)
    for sample_type, sample_result in result_long.groupby(SAMPLE_COL, dropna=False, sort=True)
}
result_wide_by_sample_type = {
    str(sample_type): sample_result.reset_index(drop=True)
    for sample_type, sample_result in result_wide.groupby(SAMPLE_COL, dropna=False, sort=True)
}

display(result_long)
display(result_wide)

,mid_sample_type,evaluation_kind,scope_type,scope_value,experiment,partner_count,n,bad_count,bad_rate,coverage,model_auc,model_ks,model_score_direction,base_auc,base_ks,base_score_direction,delta_auc,delta_ks
0,1-train,diagnostic_non_oos,overall,1-train,S1_train_only_unweighted,6,7.350170e+05,18755.000000,0.025516,1.000000,0.654063,0.216468,higher_is_bad,0.637756,0.194369,higher_is_bad,0.016307,0.022099
1,1-train,diagnostic_non_oos,overall,1-train,S1_train_only_weighted,6,7.350170e+05,18755.000000,0.025516,1.000000,0.631754,0.186643,higher_is_bad,0.637756,0.194369,higher_is_bad,-0.006003,-0.007725
2,1-train,diagnostic_non_oos,overall,1-train,S2_train_plus_ensemble_zhongyin_unweighted,6,7.350170e+05,18755.000000,0.025516,1.000000,0.617464,0.168124,higher_is_bad,0.637756,0.194369,higher_is_bad,-0.020293,-0.026245
3,1-train,diagnostic_non_oos,overall,1-train,S2_train_plus_ensemble_zhongyin_weighted,6,7.350170e+05,18755.000000,0.025516,1.000000,0.632513,0.183817,higher_is_bad,0.637756,0.194369,higher_is_bad,-0.005243,-0.010552
4,1-train,diagnostic_non_oos,overall,1-train,S3_train_plus_ensemble_non_zhongyin_unweighted,6,7.350170e+05,18755.000000,0.025516,1.000000,0.623064,0.171492,higher_is_bad,0.637756,0.194369,higher_is_bad,-0.014692,-0.022877
5,1-train,diagnostic_non_oos,overall,1-train,S3_train_plus_ensemble_non_zhongyin_weighted,6,7.350170e+05,18755.000000,0.025516,1.000000,0.632307,0.184819,higher_is_bad,0.637756,0.194369,higher_is_bad,-0.005449,-0.009549
6,1-train,diagnostic_non_oos,overall,1-train,S4_train_plus_all_ensemble_unweighted,6,7.350170e+05,18755.000000,0.025516,1.000000,0.621027,0.168928,higher_is_bad,0.637756,0.194369,higher_is_bad,-0.016729,-0.025441
7,1-train,diagnostic_non_oos,overall,1-train,S4_train_plus_all_ensemble_weighted,6,7.350170e+05,18755.000000,0.025516,1.000000,0.633723,0.187706,higher_is_bad,0.637756,0.194369,higher_is_bad,-0.004034,-0.006663
8,1-train,diagnostic_non_oos,mean,partner_macro_mean,S1_train_only_unweighted,6,1.225028e+05,3125.833333,0.033114,1.000000,0.656897,0.230328,higher_is_bad,0.636757,0.200408,higher_is_bad,0.020140,0.029920
9,1-train,diagnostic_non_oos,mean,partner_macro_mean,S1_train_only_weighted,6,1.225028e+05,3125.833333,0.033114,1.000000,0.703682,0.305485,higher_is_bad,0.636757,0.200408,higher_is_bad,0.066925,0.105077


,mid_sample_type,evaluation_kind,scope_type,scope_value,metric,Base,S1_train_only_unweighted,S1_train_only_weighted,S2_train_plus_ensemble_zhongyin_unweighted,S2_train_plus_ensemble_zhongyin_weighted,S3_train_plus_ensemble_non_zhongyin_unweighted,S3_train_plus_ensemble_non_zhongyin_weighted,S4_train_plus_all_ensemble_unweighted,S4_train_plus_all_ensemble_weighted
0,1-train,diagnostic_non_oos,overall,1-train,auc,6.377565e-01,6.540630e-01,6.317540e-01,6.174639e-01,6.325132e-01,6.230640e-01,6.323071e-01,6.210274e-01,6.337228e-01
1,1-train,diagnostic_non_oos,overall,1-train,bad_count,1.875500e+04,1.875500e+04,1.875500e+04,1.875500e+04,1.875500e+04,1.875500e+04,1.875500e+04,1.875500e+04,1.875500e+04
2,1-train,diagnostic_non_oos,overall,1-train,bad_rate,2.551642e-02,2.551642e-02,2.551642e-02,2.551642e-02,2.551642e-02,2.551642e-02,2.551642e-02,2.551642e-02,2.551642e-02
3,1-train,diagnostic_non_oos,overall,1-train,coverage,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
4,1-train,diagnostic_non_oos,overall,1-train,ks,1.943688e-01,2.164676e-01,1.866434e-01,1.681237e-01,1.838169e-01,1.714922e-01,1.848194e-01,1.689282e-01,1.877060e-01
5,1-train,diagnostic_non_oos,overall,1-train,n,7.350170e+05,7.350170e+05,7.350170e+05,7.350170e+05,7.350170e+05,7.350170e+05,7.350170e+05,7.350170e+05,7.350170e+05
6,1-train,diagnostic_non_oos,overall,1-train,partner_count,6.000000e+00,6.000000e+00,6.000000e+00,6.000000e+00,6.000000e+00,6.000000e+00,6.000000e+00,6.000000e+00,6.000000e+00
7,1-train,diagnostic_non_oos,mean,partner_macro_mean,auc,6.367571e-01,6.568968e-01,7.036816e-01,6.192176e-01,6.994759e-01,6.165635e-01,6.903785e-01,6.155883e-01,6.904619e-01
8,1-train,diagnostic_non_oos,mean,partner_macro_mean,bad_count,3.125833e+03,3.125833e+03,3.125833e+03,3.125833e+03,3.125833e+03,3.125833e+03,3.125833e+03,3.125833e+03,3.125833e+03
9,1-train,diagnostic_non_oos,mean,partner_macro_mean,bad_rate,3.311431e-02,3.311431e-02,3.311431e-02,3.311431e-02,3.311431e-02,3.311431e-02,3.311431e-02,3.311431e-02,3.311431e-02


In [49]:
result_wide.to_excel('2-建模结果.xlsx', index=False)

In [50]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve

def binary_metrics(y_true, score):
    """使用原始分数计算 AUC/KS，不自动执行 1-score。"""
    y = pd.to_numeric(pd.Series(y_true), errors='coerce')
    s = pd.to_numeric(pd.Series(score), errors='coerce')

    valid = (
        y.isin([0, 1])
        & y.notna()
        & s.notna()
        & np.isfinite(s)
    )
    y = y.loc[valid].astype(int)
    s = s.loc[valid].astype(float)

    if y.empty or y.nunique() < 2:
        return {
            'auc': np.nan,
            'ks': np.nan,
            'score_direction': '不可判断'
        }

    auc = float(roc_auc_score(y, s))
    fpr, tpr, _ = roc_curve(y, s)

    return {
        'auc': auc,
        'ks': float(np.max(np.abs(tpr - fpr))),
        'score_direction': (
            'higher_is_bad' if auc >= 0.5 else 'lower_is_bad'
        )
    }

def compare_model_with_base(
    eval_df,
    pred_col,
    label_col='flag',
    base_col='蜜蜂分_中利率版6_2_proba'
):
    """在标签、Base、模型分数均有效的共同样本上比较。"""
    label = pd.to_numeric(eval_df[label_col], errors='coerce')
    base_score = pd.to_numeric(eval_df[base_col], errors='coerce')
    model_score = pd.to_numeric(eval_df[pred_col], errors='coerce')

    labeled_mask = label.isin([0, 1])
    common_mask = (
        labeled_mask
        & base_score.notna()
        & model_score.notna()
        & np.isfinite(base_score)
        & np.isfinite(model_score)
    )

    common_df = eval_df.loc[common_mask]
    model_metrics = binary_metrics(
        common_df[label_col],
        common_df[pred_col]
    )
    base_metrics = binary_metrics(
        common_df[label_col],
        common_df[base_col]
    )

    n = int(len(common_df))
    labeled_n = int(labeled_mask.sum())
    bad_count = int(common_df[label_col].sum()) if n else 0

    return {
        'n': n,
        'bad_count': bad_count,
        'bad_rate': (
            float(common_df[label_col].mean()) if n else np.nan
        ),
        'coverage': n / labeled_n if labeled_n else np.nan,

        'model_auc': model_metrics['auc'],
        'model_ks': model_metrics['ks'],
        'model_score_direction': model_metrics['score_direction'],

        'base_auc': base_metrics['auc'],
        'base_ks': base_metrics['ks'],
        'base_score_direction': base_metrics['score_direction'],

        'delta_auc': model_metrics['auc'] - base_metrics['auc'],
        'delta_ks': model_metrics['ks'] - base_metrics['ks']
    }

def build_train_result_tables(
    model_df,
    prediction_df,
    experiment_registry,
    train_scenarios,
    id_col='test_id',
    label_col='flag',
    base_col='蜜蜂分_中利率版6_2_proba',
    partner_col='partner_name'
):
    """
    按八组实验各自的实际训练范围计算结果。

    参数
    ----
    model_df:
        完整建模数据。

    prediction_df:
        至少包含 test_id 和八个实验预测列。
        预测列名必须与 experiment_registry['experiment'] 一致。

    experiment_registry:
        至少包含 experiment、scenario 两列。

    train_scenarios:
        TRAIN_SCENARIOS 字典：
        {
            scenario_name: 与 model_df.index 对齐的布尔 Series
        }

    返回
    ----
    result_long:
        包含 overall、mean、partner_name 明细。
        mean 为各 partner_name 的 AUC/KS 宏平均，不按样本量加权。

    result_wide:
        多级列宽表，第一层为指标，第二层为八组实验。
    """
    required_registry_cols = {'experiment', 'scenario'}
    missing_registry_cols = (
        required_registry_cols - set(experiment_registry.columns)
    )
    if missing_registry_cols:
        raise KeyError(
            'experiment_registry 缺少字段: {}'.format(
                sorted(missing_registry_cols)
            )
        )

    registry = (
        experiment_registry[['experiment', 'scenario']]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    if len(registry) != 8:
        raise ValueError(
            '必须有八组实验，当前为 {} 组'.format(len(registry))
        )

    if registry['experiment'].duplicated().any():
        raise ValueError('experiment_registry 中 experiment 不唯一')

    experiment_order = registry['experiment'].tolist()

    required_prediction_cols = {id_col} | set(experiment_order)
    missing_prediction_cols = (
        required_prediction_cols - set(prediction_df.columns)
    )
    if missing_prediction_cols:
        raise KeyError(
            'prediction_df 缺少字段: {}'.format(
                sorted(missing_prediction_cols)
            )
        )

    if model_df[id_col].isna().any():
        raise ValueError('model_df.test_id 存在空值')

    if model_df[id_col].duplicated().any():
        raise ValueError('model_df.test_id 存在重复值')

    if prediction_df[id_col].isna().any():
        raise ValueError('prediction_df.test_id 存在空值')

    if prediction_df[id_col].duplicated().any():
        raise ValueError('prediction_df.test_id 存在重复值')

    prediction_index = (
        prediction_df[[id_col] + experiment_order]
        .set_index(id_col)
    )

    result_rows = []
    partner_detail_rows = []

    macro_metric_cols = [
        'model_auc',
        'model_ks',
        'base_auc',
        'base_ks'
    ]

    for registry_row in registry.itertuples(index=False):
        experiment_name = registry_row.experiment
        scenario_name = registry_row.scenario

        if scenario_name not in train_scenarios:
            raise KeyError(
                'TRAIN_SCENARIOS 中不存在 {}'.format(scenario_name)
            )

        train_mask = pd.Series(
            train_scenarios[scenario_name],
            index=model_df.index
        ).fillna(False).astype(bool)

        train_df = model_df.loc[
            train_mask,
            [
                id_col,
                label_col,
                base_col,
                partner_col
            ]
        ].copy()

        if train_df.empty:
            raise ValueError(
                '{} 的实际训练集为空'.format(experiment_name)
            )

        train_df[experiment_name] = (
            prediction_index[experiment_name]
            .reindex(train_df[id_col])
            .to_numpy()
        )

        # 1. 实际训练集整体指标
        overall_metrics = compare_model_with_base(
            eval_df=train_df,
            pred_col=experiment_name,
            label_col=label_col,
            base_col=base_col
        )

        overall_row = {
            'experiment': experiment_name,
            'scenario': scenario_name,
            'scope_type': 'overall',
            'scope_value': 'actual_train',
            'partner_count': int(
                train_df[partner_col].nunique(dropna=False)
            )
        }
        overall_row.update(overall_metrics)
        result_rows.append(overall_row)

        # 2. 各 partner_name 指标
        current_partner_rows = []

        for partner_name, partner_df in train_df.groupby(
            partner_col,
            dropna=False,
            sort=True
        ):
            partner_metrics = compare_model_with_base(
                eval_df=partner_df,
                pred_col=experiment_name,
                label_col=label_col,
                base_col=base_col
            )

            partner_row = {
                'experiment': experiment_name,
                'scenario': scenario_name,
                'scope_type': 'partner_name',
                'scope_value': str(partner_name),
                'partner_count': 1
            }
            partner_row.update(partner_metrics)

            current_partner_rows.append(partner_row)
            partner_detail_rows.append(partner_row)
            result_rows.append(partner_row)

        # 3. partner_name 宏平均
        partner_result = pd.DataFrame(current_partner_rows)
        macro_metrics = (
            partner_result[macro_metric_cols]
            .mean(skipna=True)
            .to_dict()
        )

        mean_row = {
            'experiment': experiment_name,
            'scenario': scenario_name,
            'scope_type': 'mean',
            'scope_value': 'partner_macro_mean',
            'partner_count': int(len(partner_result)),

            # mean 行的样本统计仍使用该实验实际训练集整体口径
            'n': overall_metrics['n'],
            'bad_count': overall_metrics['bad_count'],
            'bad_rate': overall_metrics['bad_rate'],
            'coverage': overall_metrics['coverage'],

            'model_auc': macro_metrics['model_auc'],
            'model_ks': macro_metrics['model_ks'],
            'base_auc': macro_metrics['base_auc'],
            'base_ks': macro_metrics['base_ks'],

            'delta_auc': (
                macro_metrics['model_auc']
                - macro_metrics['base_auc']
            ),
            'delta_ks': (
                macro_metrics['model_ks']
                - macro_metrics['base_ks']
            ),

            'model_score_direction': 'partner_macro_mean',
            'base_score_direction': 'partner_macro_mean'
        }
        result_rows.append(mean_row)

    result_long = pd.DataFrame(result_rows)

    scope_order = {
        'overall': 0,
        'mean': 1,
        'partner_name': 2
    }
    experiment_order_map = {
        name: index
        for index, name in enumerate(experiment_order)
    }

    result_long['_experiment_order'] = (
        result_long['experiment'].map(experiment_order_map)
    )
    result_long['_scope_order'] = (
        result_long['scope_type'].map(scope_order)
    )

    result_long = (
        result_long
        .sort_values(
            [
                '_scope_order',
                'scope_value',
                '_experiment_order'
            ]
        )
        .drop(
            columns=[
                '_scope_order',
                '_experiment_order'
            ]
        )
        .reset_index(drop=True)
    )

    wide_metric_cols = [
        'partner_count',
        'n',
        'bad_count',
        'bad_rate',
        'coverage',
        'model_auc',
        'base_auc',
        'delta_auc',
        'model_ks',
        'base_ks',
        'delta_ks'
    ]

    result_wide = (
        result_long
        .pivot(
            index=['scope_type', 'scope_value'],
            columns='experiment',
            values=wide_metric_cols
        )
        .reindex(
            columns=experiment_order,
            level='experiment'
        )
    )

    result_wide.columns.names = ['metric', 'experiment']
    result_wide = result_wide.reset_index()

    return result_long, result_wide

# ============================================================
# 训练时需要保证 prediction_df 包含八组实验在各自实际训练集上的预测
# ============================================================

# 在八组训练循环开始前初始化：
#
# prediction_df = model_df[['test_id']].copy()
# for experiment_name in experiment_order:
#     prediction_df[experiment_name] = np.nan
#
# 在每组 experiment_model 训练完成后执行：
#
# train_pred = experiment_model.predict_proba(
#     scenario_df[feature_cols]
# )[:, 1]
#
# prediction_df.loc[
#     scenario_df.index,
#     experiment_name
# ] = train_pred

# ============================================================
# 复用方式
# ============================================================



In [51]:
prediction_df.head()

,test_id,flag,partner_name,partner_code,mid_sample_type,ym_proba_a077_3,S1_train_only_unweighted,S1_train_only_weighted,S2_train_plus_ensemble_zhongyin_unweighted,S2_train_plus_ensemble_zhongyin_weighted,S3_train_plus_ensemble_non_zhongyin_unweighted,S3_train_plus_ensemble_non_zhongyin_weighted,S4_train_plus_all_ensemble_unweighted,S4_train_plus_all_ensemble_weighted
861159,4491235,0.0,lexin_type2,c_19,3-oos,0.015791,0.017216,0.018238,0.023072,0.018793,0.049208,0.014765,0.046420,0.014420
861160,4491238,0.0,lexin_type2,c_19,3-oos,0.022935,0.020459,0.023324,0.036012,0.027657,0.062692,0.026999,0.064484,0.027680
861161,4491240,0.0,lexin_type2,c_19,3-oos,0.030106,0.029071,0.038350,0.043219,0.037702,0.046265,0.037241,0.058370,0.038571
861162,4491242,0.0,lexin_type2,c_19,3-oos,0.050666,0.031867,0.046754,0.065616,0.047111,0.065718,0.046410,0.078471,0.047358
861163,4491248,0.0,lexin_type2,c_19,3-oos,0.040962,0.034504,0.066152,0.054586,0.057329,0.053201,0.060547,0.070170,0.061524


In [52]:
experiment_registry

,experiment,scenario,weighted,train_n,train_bad_count,train_bad_rate,base_cnt,base_rate
0,S1_train_only_unweighted,S1_train_only,False,735017,18755,0.025516,NaN,NaN
1,S1_train_only_weighted,S1_train_only,True,735017,18755,0.025516,30000.0,0.04
2,S2_train_plus_ensemble_zhongyin_unweighted,S2_train_plus_ensemble_zhongyin,False,1095017,53755,0.049091,NaN,NaN
3,S2_train_plus_ensemble_zhongyin_weighted,S2_train_plus_ensemble_zhongyin,True,1095017,53755,0.049091,30000.0,0.04
4,S3_train_plus_ensemble_non_zhongyin_unweighted,S3_train_plus_ensemble_non_zhongyin,False,1195645,62094,0.051933,NaN,NaN
5,S3_train_plus_ensemble_non_zhongyin_weighted,S3_train_plus_ensemble_non_zhongyin,True,1195645,62094,0.051933,30000.0,0.04
6,S4_train_plus_all_ensemble_unweighted,S4_train_plus_all_ensemble,False,1555645,97094,0.062414,NaN,NaN
7,S4_train_plus_all_ensemble_weighted,S4_train_plus_all_ensemble,True,1555645,97094,0.062414,30000.0,0.04


In [55]:
scenario_df

,test_id,partner_code,flag,partner_name,mid_sample_type,ym_proba_a077_3,hxe1,hxe2,hxe3,hxe4,hxe5,hxe6,hxe7,hxe8,hxe9,hxe10,hxe11,hxe12,hxe13,hxe14,hxe15,hxe16,hxe17,hxe18,hxe19,hxe20,hxe21,hxe22,hxe23,hxe24,hxe25,hxe26,hxe27,hxe28,hxe29,hxe30,hxe31,hxe32,hxe33,hxe34,hxe35,hxe36,hxe37,hxe38,hxe39,hxe40,hxe41,hxe42,hxe43,hxe44,hxe45,hxe46,hxe47,hxe48,hxe49,hxe50,hxe51,hxe52,hxe53,hxe54,hxe55,hxe56,hxe57,hxe58,hxe59,hxe60,hxe61,hxe62,hxe63,hxe64,hxe65,hxe66,hxe67,hxe68,hxe69,hxe70,hxe71,hxe72,hxe73,hxe74,hxe75,hxe76,hxe77,hxe78,hxe79,hxe80,hce1,hce10,hce11,hce12,hce13,hce14,hce15,hce16,hce17,hce18,hce19,hce2,hce20,hce21,...,hcu12,hcu120,hcu121,hcu122,hcu13,hcu14,hcu15,hcu16,hcu17,hcu18,hcu19,hcu2,hcu20,hcu21,hcu22,hcu23,hcu24,hcu25,hcu26,hcu27,hcu28,hcu29,hcu3,hcu30,hcu31,hcu32,hcu33,hcu34,hcu35,hcu36,hcu37,hcu38,hcu39,hcu4,hcu40,hcu41,hcu42,hcu43,hcu44,hcu45,hcu46,hcu47,hcu48,hcu49,hcu5,hcu50,hcu51,hcu52,hcu53,hcu54,hcu55,hcu56,hcu57,hcu58,hcu59,hcu6,hcu60,hcu61,hcu62,hcu63,hcu64,hcu65,hcu66,hcu67,hcu68,hcu69,hcu7,hcu70,hcu71,hcu72,hcu73,hcu74,hcu75,hcu76,hcu77,hcu78,hcu79,hcu8,hcu80,hcu81,hcu82,hcu83,hcu84,hcu85,hcu86,hcu87,hcu88,hcu89,hcu9,hcu90,hcu91,hcu92,hcu93,hcu94,hcu95,hcu96,hcu97,hcu98,hcu99,weight_group
1099145,5078698,c_22,0.0,lexin_type5,1-train,0.016213,0.368965,0.0,0.738302,0.210272,0.169975,0.0,0.0,0.0,0.410020,0.0,0.110803,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.097291,0.439660,0.0,0.090329,0.0,0.0,0.0,0.0,0.293960,0.0,0.367908,0.0,0.0,0.0,0.0,0.634632,0.0,0.0,0.0,0.0,0.0,0.225145,0.0,0.0,0.0,0.0,0.064608,0.327991,0.411153,0.472463,0.734618,0.0,0.020178,0.275875,0.176588,0.0,0.637387,0.0,0.0,0.065354,0.0,0.0,0.635983,0.0,0.0,0.0,0.0,0.0,0.660666,0.277334,0.221372,0.651886,0.024861,0.519368,0.0,0.419697,0.0,0.812967,0.0,0.0,0.0,0.0,1.0,19.0,23.0,23.0,1.0,23.0,24.0,23.0,13.0,23.0,22.0,23.0,14.0,4.0,...,0.0670,0.7212,2.2600,0.0,0.0688,0.2911,0.1916,0.7212,0.1163,0.1346,0.0025,0.0561,0.0591,0.0160,0.0061,0.0037,0.0018,0.3225,0.1369,0.2047,0.3358,0.5061,0.4213,0.4369,0.3385,0.2246,0.3156,0.1776,0.2566,0.2571,0.2600,0.2263,0.1428,0.1262,0.1749,0.4015,0.6712,0.1242,0.0466,0.0361,0.5206,0.0316,0.0057,0.5282,0.1607,0.4718,0.4825,0.5175,0.2279,0.7721,0.9716,0.0284,0.8500,0.1500,0.7311,0.2229,0.2689,0.6090,0.3910,0.1783,0.8217,0.3871,-2.9125,-0.6329,1.3826,0.1773,0.0686,-1.9657,0.0860,-1.0916,-4.2955,0.1296,-0.0595,-0.8504,0.9798,0.4593,0.3371,0.0028,-2.9485,2.1103,0.8255,0.2107,-1.0943,-2.0676,-3.2923,-1.4983,-2.3554,-0.4209,0.1061,0.8084,0.1431,2.1846,1.6094,1.9599,1.8342,-0.6006,-0.3977,0.0757,0.4334,1-train_c_22
1099146,5078699,c_22,0.0,lexin_type5,1-train,0.033341,0.133713,0.0,0.525854,0.246294,0.071848,0.0,0.0,0.0,0.368628,0.0,0.167582,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.131253,0.658361,0.0,0.113791,0.0,0.0,0.0,0.0,0.290049,0.0,0.086355,0.0,0.0,0.0,0.0,0.453155,0.0,0.0,0.0,0.0,0.0,0.223154,0.0,0.0,0.0,0.0,0.107995,0.013784,0.395417,0.147107,0.543833,0.0,0.058792,0.269772,0.060938,0.0,0.608494,0.0,0.0,0.129593,0.0,0.0,0.472863,0.0,0.0,0.0,0.0,0.0,0.499968,0.371330,0.293696,0.751994,0.000000,0.491910,0.0,0.423398,0.0,0.745430,0.0,0.0,0.0,0.0,1.0,1.0,7.0,1.0,20.0,1.0,8.0,6.0,15.0,6.0,8.0,1.0,11.0,17.0,...,0.0244,0.8438,1.2811,1.0,0.0455,0.3017,0.1214,0.8438,0.0855,0.0443,0.0029,0.1189,0.0339,0.0131,0.0050,0.0031,0.0014,0.3027,0.1634,0.2474,0.2865,0.1937,0.2475,0.4456,0.3272,0.2272,0.2479,0.1733,0.2444,0.2811,0.2528,0.2217,0.5286,0.2426,0.2542,0.0746,0.2495,0.4042,0.0448,0.1939,0.0205,0.0336,0.0199,0.4679,0.1681,0.5321,0.7845,0.2155,0.3254,0.6746,0.8449,0.1551,0.3540,0.6460,0.8337,0.0954,0.1663,0.5983,0.4017,0.7489,0.2511,0.7113,-1.9749,-0.5095,0.2235,0.2032,0.0163,-0.5892,-0.2043,-1.9734,-4.4010,0.1810,0.1006,-0.7926,0.7315,-0.5270,-0.2576,0.0014,-3.0793,1.8054,0.4580,-0.6374,-1.5863,-2.5437,-3.8023,0.5073,-0.1094,0.8190,0.1609,0.8925,0.2189,0.3667,0.0084,0.1146,0.0171,1.5707,0.8385,0.2607,-0.3869,1-train_c_22
1099147,5078701,c_22,0.0,lexin_type5,1-train,0.023149,0.000000,0.0,0.142918,0.198639,0.000000,0.0,0.0,0.0,0.311823,0.0

In [57]:
TRAIN_SCENARIOS[row.scenario]

861159     False
861160     False
861161     False
861162     False
861163     False
           ...  
4353499     True
4353500     True
4353501     True
4353502     True
4353503     True
Length: 3181655, dtype: bool

In [59]:
scenario_df.head()

,test_id,partner_code,flag,partner_name,mid_sample_type,ym_proba_a077_3,hxe1,hxe2,hxe3,hxe4,hxe5,hxe6,hxe7,hxe8,hxe9,hxe10,hxe11,hxe12,hxe13,hxe14,hxe15,hxe16,hxe17,hxe18,hxe19,hxe20,hxe21,hxe22,hxe23,hxe24,hxe25,hxe26,hxe27,hxe28,hxe29,hxe30,hxe31,hxe32,hxe33,hxe34,hxe35,hxe36,hxe37,hxe38,hxe39,hxe40,hxe41,hxe42,hxe43,hxe44,hxe45,hxe46,hxe47,hxe48,hxe49,hxe50,hxe51,hxe52,hxe53,hxe54,hxe55,hxe56,hxe57,hxe58,hxe59,hxe60,hxe61,hxe62,hxe63,hxe64,hxe65,hxe66,hxe67,hxe68,hxe69,hxe70,hxe71,hxe72,hxe73,hxe74,hxe75,hxe76,hxe77,hxe78,hxe79,hxe80,hce1,hce10,hce11,hce12,hce13,hce14,hce15,hce16,hce17,hce18,hce19,hce2,hce20,hce21,...,hcu12,hcu120,hcu121,hcu122,hcu13,hcu14,hcu15,hcu16,hcu17,hcu18,hcu19,hcu2,hcu20,hcu21,hcu22,hcu23,hcu24,hcu25,hcu26,hcu27,hcu28,hcu29,hcu3,hcu30,hcu31,hcu32,hcu33,hcu34,hcu35,hcu36,hcu37,hcu38,hcu39,hcu4,hcu40,hcu41,hcu42,hcu43,hcu44,hcu45,hcu46,hcu47,hcu48,hcu49,hcu5,hcu50,hcu51,hcu52,hcu53,hcu54,hcu55,hcu56,hcu57,hcu58,hcu59,hcu6,hcu60,hcu61,hcu62,hcu63,hcu64,hcu65,hcu66,hcu67,hcu68,hcu69,hcu7,hcu70,hcu71,hcu72,hcu73,hcu74,hcu75,hcu76,hcu77,hcu78,hcu79,hcu8,hcu80,hcu81,hcu82,hcu83,hcu84,hcu85,hcu86,hcu87,hcu88,hcu89,hcu9,hcu90,hcu91,hcu92,hcu93,hcu94,hcu95,hcu96,hcu97,hcu98,hcu99,weight_group
1099145,5078698,c_22,0.0,lexin_type5,1-train,0.016213,0.368965,0.0,0.738302,0.210272,0.169975,0.0,0.0,0.0,0.410020,0.0,0.110803,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.097291,0.439660,0.0,0.090329,0.0,0.0,0.0,0.0,0.293960,0.0,0.367908,0.0,0.0,0.0,0.0,0.634632,0.0,0.0,0.0,0.0,0.0,0.225145,0.0,0.0,0.0,0.0,0.064608,0.327991,0.411153,0.472463,0.734618,0.0,0.020178,0.275875,0.176588,0.0,0.637387,0.0,0.0,0.065354,0.0,0.0,0.635983,0.0,0.0,0.0,0.0,0.0,0.660666,0.277334,0.221372,0.651886,0.024861,0.519368,0.0,0.419697,0.0,0.812967,0.0,0.0,0.0,0.0,1.0,19.0,23.0,23.0,1.0,23.0,24.0,23.0,13.0,23.0,22.0,23.0,14.0,4.0,...,0.0670,0.7212,2.2600,0.0,0.0688,0.2911,0.1916,0.7212,0.1163,0.1346,0.0025,0.0561,0.0591,0.0160,0.0061,0.0037,0.0018,0.3225,0.1369,0.2047,0.3358,0.5061,0.4213,0.4369,0.3385,0.2246,0.3156,0.1776,0.2566,0.2571,0.2600,0.2263,0.1428,0.1262,0.1749,0.4015,0.6712,0.1242,0.0466,0.0361,0.5206,0.0316,0.0057,0.5282,0.1607,0.4718,0.4825,0.5175,0.2279,0.7721,0.9716,0.0284,0.8500,0.1500,0.7311,0.2229,0.2689,0.6090,0.3910,0.1783,0.8217,0.3871,-2.9125,-0.6329,1.3826,0.1773,0.0686,-1.9657,0.0860,-1.0916,-4.2955,0.1296,-0.0595,-0.8504,0.9798,0.4593,0.3371,0.0028,-2.9485,2.1103,0.8255,0.2107,-1.0943,-2.0676,-3.2923,-1.4983,-2.3554,-0.4209,0.1061,0.8084,0.1431,2.1846,1.6094,1.9599,1.8342,-0.6006,-0.3977,0.0757,0.4334,1-train_c_22
1099146,5078699,c_22,0.0,lexin_type5,1-train,0.033341,0.133713,0.0,0.525854,0.246294,0.071848,0.0,0.0,0.0,0.368628,0.0,0.167582,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.131253,0.658361,0.0,0.113791,0.0,0.0,0.0,0.0,0.290049,0.0,0.086355,0.0,0.0,0.0,0.0,0.453155,0.0,0.0,0.0,0.0,0.0,0.223154,0.0,0.0,0.0,0.0,0.107995,0.013784,0.395417,0.147107,0.543833,0.0,0.058792,0.269772,0.060938,0.0,0.608494,0.0,0.0,0.129593,0.0,0.0,0.472863,0.0,0.0,0.0,0.0,0.0,0.499968,0.371330,0.293696,0.751994,0.000000,0.491910,0.0,0.423398,0.0,0.745430,0.0,0.0,0.0,0.0,1.0,1.0,7.0,1.0,20.0,1.0,8.0,6.0,15.0,6.0,8.0,1.0,11.0,17.0,...,0.0244,0.8438,1.2811,1.0,0.0455,0.3017,0.1214,0.8438,0.0855,0.0443,0.0029,0.1189,0.0339,0.0131,0.0050,0.0031,0.0014,0.3027,0.1634,0.2474,0.2865,0.1937,0.2475,0.4456,0.3272,0.2272,0.2479,0.1733,0.2444,0.2811,0.2528,0.2217,0.5286,0.2426,0.2542,0.0746,0.2495,0.4042,0.0448,0.1939,0.0205,0.0336,0.0199,0.4679,0.1681,0.5321,0.7845,0.2155,0.3254,0.6746,0.8449,0.1551,0.3540,0.6460,0.8337,0.0954,0.1663,0.5983,0.4017,0.7489,0.2511,0.7113,-1.9749,-0.5095,0.2235,0.2032,0.0163,-0.5892,-0.2043,-1.9734,-4.4010,0.1810,0.1006,-0.7926,0.7315,-0.5270,-0.2576,0.0014,-3.0793,1.8054,0.4580,-0.6374,-1.5863,-2.5437,-3.8023,0.5073,-0.1094,0.8190,0.1609,0.8925,0.2189,0.3667,0.0084,0.1146,0.0171,1.5707,0.8385,0.2607,-0.3869,1-train_c_22
1099147,5078701,c_22,0.0,lexin_type5,1-train,0.023149,0.000000,0.0,0.142918,0.198639,0.000000,0.0,0.0,0.0,0.311823,0.0,0.157168,

In [58]:
row

experiment         S4_train_plus_all_ensemble_weighted
scenario                    S4_train_plus_all_ensemble
weighted                                          True
train_n                                        1555645
train_bad_count                                  97094
train_bad_rate                                0.062414
base_cnt                                       30000.0
base_rate                                         0.04
Name: 7, dtype: object

In [60]:
for i, row in experiment_registry.iterrows():
    print(i, row.experiment, row.scenario)
    # 计算每组实验的实际训练样本的binary_metrics
    train_mask = TRAIN_SCENARIOS[row.scenario]
    # train_mask剔除中银
    train_mask = train_mask & (scenario_df['partner_name'] != '中银')
    print(binary_metrics(
        scenario_df.loc[train_mask, 'flag'],
        prediction_df.loc[train_mask, row.experiment]
    ))



0 S1_train_only_unweighted S1_train_only
{'auc': 0.6540629877656526, 'ks': 0.21646761491305583, 'score_direction': 'higher_is_bad'}
1 S1_train_only_weighted S1_train_only
{'auc': 0.6317539679574989, 'ks': 0.18664336370435264, 'score_direction': 'higher_is_bad'}
2 S2_train_plus_ensemble_zhongyin_unweighted S2_train_plus_ensemble_zhongyin
{'auc': 0.6174638688429187, 'ks': 0.16812365472032031, 'score_direction': 'higher_is_bad'}
3 S2_train_plus_ensemble_zhongyin_weighted S2_train_plus_ensemble_zhongyin
{'auc': 0.6325132186888616, 'ks': 0.18381691583181664, 'score_direction': 'higher_is_bad'}
4 S3_train_plus_ensemble_non_zhongyin_unweighted S3_train_plus_ensemble_non_zhongyin
{'auc': 0.695359504316753, 'ks': 0.29011627312977567, 'score_direction': 'higher_is_bad'}
5 S3_train_plus_ensemble_non_zhongyin_weighted S3_train_plus_ensemble_non_zhongyin
{'auc': 0.6298627160734664, 'ks': 0.18464035108607585, 'score_direction': 'higher_is_bad'}
6 S4_train_plus_all_ensemble_unweighted S4_train_plus_a

In [53]:

result_long, result_wide = build_train_result_tables(
    model_df=model_df,
    prediction_df=prediction_df,
    experiment_registry=experiment_registry,
    train_scenarios=TRAIN_SCENARIOS,
    id_col='test_id',
    label_col='flag',
    base_col='ym_proba_a077_3',
    partner_col='partner_name'
)

display(result_long)
display(result_wide)

# 只查看 overall 和 mean
result_long_summary = result_long.loc[
    result_long['scope_type'].isin(['overall', 'mean'])
].reset_index(drop=True)

result_wide_summary = result_wide.loc[
    result_wide['scope_type'].isin(['overall', 'mean'])
].reset_index(drop=True)

display(result_long_summary)
display(result_wide_summary)

,experiment,scenario,scope_type,scope_value,partner_count,n,bad_count,bad_rate,coverage,model_auc,model_ks,model_score_direction,base_auc,base_ks,base_score_direction,delta_auc,delta_ks
0,S1_train_only_unweighted,S1_train_only,overall,actual_train,6,735017,18755,0.025516,1.000000,0.654063,0.216468,higher_is_bad,0.637756,0.194369,higher_is_bad,0.016307,0.022099
1,S1_train_only_weighted,S1_train_only,overall,actual_train,6,735017,18755,0.025516,1.000000,0.631754,0.186643,higher_is_bad,0.637756,0.194369,higher_is_bad,-0.006003,-0.007725
2,S2_train_plus_ensemble_zhongyin_unweighted,S2_train_plus_ensemble_zhongyin,overall,actual_train,7,1084406,52646,0.048548,0.990310,0.677367,0.257318,higher_is_bad,0.603620,0.148010,higher_is_bad,0.073747,0.109307
3,S2_train_plus_ensemble_zhongyin_weighted,S2_train_plus_ensemble_zhongyin,overall,actual_train,7,1084406,52646,0.048548,0.990310,0.597621,0.137740,higher_is_bad,0.603620,0.148010,higher_is_bad,-0.005999,-0.010270
4,S3_train_plus_ensemble_non_zhongyin_unweighted,S3_train_plus_ensemble_non_zhongyin,overall,actual_train,9,1195645,62094,0.051933,1.000000,0.695360,0.290116,higher_is_bad,0.661293,0.234421,higher_is_bad,0.034066,0.055696
5,S3_train_plus_ensemble_non_zhongyin_weighted,S3_train_plus_ensemble_non_zhongyin,overall,actual_train,9,1195645,62094,0.051933,1.000000,0.629863,0.184640,higher_is_bad,0.661293,0.234421,higher_is_bad,-0.031430,-0.049780
6,S4_train_plus_all_ensemble_unweighted,S4_train_plus_all_ensemble,overall,actual_train,10,1545034,95985,0.062125,0.993179,0.665111,0.239670,higher_is_bad,0.631914,0.190252,higher_is_bad,0.033196,0.049418
7,S4_train_plus_all_ensemble_weighted,S4_train_plus_all_ensemble,overall,actual_train,10,1545034,95985,0.062125,0.993179,0.612633,0.161055,higher_is_bad,0.631914,0.190252,higher_is_bad,-0.019281,-0.029197
8,S1_train_only_unweighted,S1_train_only,mean,partner_macro_mean,6,735017,18755,0.025516,1.000000,0.656897,0.230328,partner_macro_mean,0.636757,0.200408,partner_macro_mean,0.020140,0.029920
9,S1_train_only_weighted,S1_train_only,mean,partner_macro_mean,6,735017,18755,0.025516,1.000000,0.703682,0.305485,partner_macro_mean,0.636757,0.200408,partner_macro_mean,0.066925,0.105077


metric        scope_type             scope_value            partner_count  \
experiment                                       S1_train_only_unweighted   
0                   mean      partner_macro_mean                      6.0   
1                overall            actual_train                      6.0   
2           partner_name             lexin_type5                      1.0   
3           partner_name  weipingfubang_lowerirr                      NaN   
4           partner_name                      中银                      NaN   
5           partner_name                  海尔(借呗)                      1.0   
6           partner_name            海尔(借呗拒绝其他通过)                      1.0   
7           partner_name                海尔(字节星选)                      1.0   
8           partner_name          海尔(字节星选拒绝其他通过)                      1.0   
9           partner_name                海尔(字节智选)                      1.0   
10          partner_name                海尔(自营贷中)                      NaN   
11          partner_name                  海尔(蚂蚁)                      NaN   

metric                                                                        \
experiment S1_train_only_weighted S2_train_plus_ensemble_zhongyin_unweighted   
0                             6.0                                        7.0   
1                             6.0                                        7.0   
2                             1.0                                        1.0   
3                             NaN                                        NaN   
4                             NaN                                        1.0   
5                             1.0                                        1.0   
6                             1.0                                        1.0   
7                             1.0                                        1.0   
8                             1.0                                        1.0   
9                             1.0                                        1.0   
10                            NaN                                        NaN   
11                            NaN                                        NaN   

metric                                               \
experiment S2_train_plus_ensemble_zhongyin_weighted   
0                                               7.0   
1                                               7.0   
2                                               1.0   
3                                               NaN   
4                                               1.0   
5                                               1.0   
6                                               1.0   
7                                               1.0   
8                                               1.0   
9                                               1.0   
10                                              NaN   
11                                              NaN   

metric                                                     \
experiment S3_train_plus_ensemble_non_zhongyin_unweighted   
0                                                     9.0   
1                                                     9.0   
2                                                     1.0   
3                                                     1.0   
4                                                     NaN   
5                                                     1.0   
6                                                     1.0   
7                                                     1.0   
8                                                     1.0   
9                                                     1.0   
10                                                    1.0   
11                                                    1.0   

metric                                                   \
experiment S3_train_plus_ensemble_non_zhongyin_weighted   
0                                                   9.0 

,experiment,scenario,scope_type,scope_value,partner_count,n,bad_count,bad_rate,coverage,model_auc,model_ks,model_score_direction,base_auc,base_ks,base_score_direction,delta_auc,delta_ks
0,S1_train_only_unweighted,S1_train_only,overall,actual_train,6,735017,18755,0.025516,1.000000,0.654063,0.216468,higher_is_bad,0.637756,0.194369,higher_is_bad,0.016307,0.022099
1,S1_train_only_weighted,S1_train_only,overall,actual_train,6,735017,18755,0.025516,1.000000,0.631754,0.186643,higher_is_bad,0.637756,0.194369,higher_is_bad,-0.006003,-0.007725
2,S2_train_plus_ensemble_zhongyin_unweighted,S2_train_plus_ensemble_zhongyin,overall,actual_train,7,1084406,52646,0.048548,0.990310,0.677367,0.257318,higher_is_bad,0.603620,0.148010,higher_is_bad,0.073747,0.109307
3,S2_train_plus_ensemble_zhongyin_weighted,S2_train_plus_ensemble_zhongyin,overall,actual_train,7,1084406,52646,0.048548,0.990310,0.597621,0.137740,higher_is_bad,0.603620,0.148010,higher_is_bad,-0.005999,-0.010270
4,S3_train_plus_ensemble_non_zhongyin_unweighted,S3_train_plus_ensemble_non_zhongyin,overall,actual_train,9,1195645,62094,0.051933,1.000000,0.695360,0.290116,higher_is_bad,0.661293,0.234421,higher_is_bad,0.034066,0.055696
5,S3_train_plus_ensemble_non_zhongyin_weighted,S3_train_plus_ensemble_non_zhongyin,overall,actual_train,9,1195645,62094,0.051933,1.000000,0.629863,0.184640,higher_is_bad,0.661293,0.234421,higher_is_bad,-0.031430,-0.049780
6,S4_train_plus_all_ensemble_unweighted,S4_train_plus_all_ensemble,overall,actual_train,10,1545034,95985,0.062125,0.993179,0.665111,0.239670,higher_is_bad,0.631914,0.190252,higher_is_bad,0.033196,0.049418
7,S4_train_plus_all_ensemble_weighted,S4_train_plus_all_ensemble,overall,actual_train,10,1545034,95985,0.062125,0.993179,0.612633,0.161055,higher_is_bad,0.631914,0.190252,higher_is_bad,-0.019281,-0.029197
8,S1_train_only_unweighted,S1_train_only,mean,partner_macro_mean,6,735017,18755,0.025516,1.000000,0.656897,0.230328,partner_macro_mean,0.636757,0.200408,partner_macro_mean,0.020140,0.029920
9,S1_train_only_weighted,S1_train_only,mean,partner_macro_mean,6,735017,18755,0.025516,1.000000,0.703682,0.305485,partner_macro_mean,0.636757,0.200408,partner_macro_mean,0.066925,0.105077


metric     scope_type         scope_value            partner_count  \
experiment                                S1_train_only_unweighted   
0                mean  partner_macro_mean                      6.0   
1             overall        actual_train                      6.0   

metric                                                                        \
experiment S1_train_only_weighted S2_train_plus_ensemble_zhongyin_unweighted   
0                             6.0                                        7.0   
1                             6.0                                        7.0   

metric                                               \
experiment S2_train_plus_ensemble_zhongyin_weighted   
0                                               7.0   
1                                               7.0   

metric                                                     \
experiment S3_train_plus_ensemble_non_zhongyin_unweighted   
0                                                     9.0   
1                                                     9.0   

metric                                                   \
experiment S3_train_plus_ensemble_non_zhongyin_weighted   
0                                                   9.0   
1                                                   9.0   

metric                                            \
experiment S4_train_plus_all_ensemble_unweighted   
0                                           10.0   
1                                           10.0   

metric                                                                n  \
experiment S4_train_plus_all_ensemble_weighted S1_train_only_unweighted   
0                                         10.0                 735017.0   
1                                         10.0                 735017.0   

metric                                                                        \
experiment S1_train_only_weighted S2_train_plus_ensemble_zhongyin_unweighted   
0                        735017.0                                  1084406.0   
1                        735017.0                                  1084406.0   

metric                                               \
experiment S2_train_plus_ensemble_zhongyin_weighted   
0                                         1084406.0   
1                                         1084406.0   

metric                                                     \
experiment S3_train_plus_ensemble_non_zhongyin_unweighted   
0                                               1195645.0   
1                                               1195645.0   

metric                                                   \
experiment S3_train_plus_ensemble_non_zhongyin_weighted   
0                                             1195645.0   
1                                             1195645.0   

metric                                            \
experiment S4_train_plus_all_ensemble_unweighted   
0                                      1545034.0   
1                                      1545034.0   

metric                                                        bad_count  \
experiment S4_train_plus_all_ensemble_weighted S1_train_only_unweighted   
0                                    1545034.0                  18755.0   
1                                    1545034.0                  18755.0   

metric                                                                        \
experiment S1_train_only_weighted S2_train_plus_ensemble_zhongyin_unweighted   
0                         18755.0                                    52646.0   
1                         18755.0                                    52646.0   

metric                                               \
experiment S2_train_plus_ensemble_zhongyin_weighted   
0                                           52646.0   
1                                           52646.0   

metric                                                     \
experiment S3_train_plus_ensemble_non_zhongyin_u

# 不固定模型参数，每组实验单独调参
"""
tuner = AntiOverfittingTuner(lgb.LGBMClassifier,'lgb')
params_grid_lgb = tuner.optimize_anti_overfitting(
    df_train[feature_cols+['weight']],
    df_train['flag'],
    weight_col = 'weight',
    n_trials=30
)
"""

In [62]:
TUNER_N_TRIALS = 30
TUNER_CV_FOLDS = 5
TUNER_N_JOBS = 1
TUNER_WEIGHT_COL = '__tuner_weight__'

TUNER_MAX_AUC_GAP = 0.05
TUNER_MAX_KS_GAP = 0.03
TUNER_PENALTY_WEIGHT = 2.0

class WeightedAntiOverfittingTuner(AntiOverfittingTuner):
    """
    weighted 实验已经传入自定义 sample_weight，
    因此固定 is_unbalance=False，避免重复处理类别不平衡。
    """

    def _suggest_parameters(self, trial):
        params = super()._suggest_parameters(trial)
        params['is_unbalance'] = False
        return params

def run_anti_overfitting_tuning(
    scenario_df,
    experiment_name,
    weighted,
    feature_cols,
    best_weight_params
):
    if scenario_df.empty:
        raise ValueError(
            '{} 调参数据为空'.format(experiment_name)
        )

    if scenario_df[SAMPLE_COL].eq('3-oos').any():
        raise AssertionError(
            '{} 调参数据包含 3-oos'.format(experiment_name)
        )

    if scenario_df[LABEL_COL].isna().any():
        raise ValueError(
            '{} 调参标签存在空值'.format(experiment_name)
        )

    validate_binary_values(
        scenario_df[LABEL_COL],
        '{}.flag'.format(experiment_name),
        require_both=True
    )

    y_tune = (
        scenario_df[LABEL_COL]
        .astype(int)
        .reset_index(drop=True)
    )

    min_class_count = int(y_tune.value_counts().min())
    if min_class_count < TUNER_CV_FOLDS:
        raise ValueError(
            '{} 最少类别样本数 {} 小于 CV 折数 {}'.format(
                experiment_name,
                min_class_count,
                TUNER_CV_FOLDS
            )
        )

    X_tune = (
        scenario_df[feature_cols]
        .reset_index(drop=True)
        .copy()
    )

    weight_col = None
    tuner_class = AntiOverfittingTuner

    if weighted:
        sample_weight = balance_df_weight(
            df=scenario_df,
            gp_col='weight_group',
            label_col=LABEL_COL,
            base_cnt=best_weight_params['base_cnt'],
            base_rate=best_weight_params['base_rate']
        )

        X_tune[TUNER_WEIGHT_COL] = (
            sample_weight
            .reindex(scenario_df.index)
            .to_numpy()
        )

        weight_col = TUNER_WEIGHT_COL
        tuner_class = WeightedAntiOverfittingTuner

    tuner = tuner_class(
        model_class=lgb.LGBMClassifier,
        model_type='lgb',
        direction='maximize',
        study_name='{}_anti_overfitting'.format(experiment_name),
        max_auc_gap=TUNER_MAX_AUC_GAP,
        max_ks_gap=TUNER_MAX_KS_GAP,
        overfitting_penalty_weight=TUNER_PENALTY_WEIGHT,
        use_regularization_focused_space=True
    )

    tuning_result = tuner.optimize_anti_overfitting(
        X=X_tune,
        y=y_tune,
        objective_function=AUCObjective(),
        n_trials=TUNER_N_TRIALS,
        cv_folds=TUNER_CV_FOLDS,
        n_jobs=TUNER_N_JOBS,
        show_progress_bar=True,
        weight_col=weight_col
    )

    # AntiOverfittingTuner 的推荐参数：
    # 优先选择同时满足 AUC gap 和 KS gap 的最佳验证集参数。
    selected_params = dict(
        tuning_result['recommended_params']
    )

    # weighted 模型禁止再次使用 LightGBM 内置类别平衡。
    if weighted:
        selected_params['is_unbalance'] = False

    performance_history = pd.DataFrame(
        tuning_result['performance_history']
    )
    performance_history.insert(
        0,
        'experiment',
        experiment_name
    )

    result_summary = tuning_result['results_summary']

    audit_row = {
        'experiment': experiment_name,
        'weighted': weighted,
        'tune_n': int(len(scenario_df)),
        'tune_bad_count': int(
            scenario_df[LABEL_COL].sum()
        ),
        'tune_bad_rate': float(
            scenario_df[LABEL_COL].mean()
        ),
        'oos_overlap_count': int(
            scenario_df[SAMPLE_COL].eq('3-oos').sum()
        ),
        'objective': 'AUCObjective',
        'n_trials': tuning_result['n_trials'],
        'cv_folds': TUNER_CV_FOLDS,
        'best_score_with_penalty':
            tuning_result['best_score'],
        'recommended_val_score':
            tuning_result['recommended_score'],
        'valid_trials':
            result_summary['valid_trials'],
        'overfitting_rate':
            result_summary['overfitting_rate'],
        'best_valid_auc_gap':
            result_summary['best_valid_auc_gap'],
        'best_valid_ks_gap':
            result_summary['best_valid_ks_gap'],
        'best_params':
            tuning_result['best_params'],
        'recommended_params':
            selected_params
    }

    del X_tune, y_tune, tuner
    gc.collect()

    return selected_params, tuning_result, performance_history, audit_row


In [63]:
tuned_params_by_experiment = {}
tuner_results_by_experiment = {}
tuner_history_parts = []
tuner_audit_rows = []

for scenario_name in [
    'S2_train_plus_ensemble_zhongyin',
    'S4_train_plus_all_ensemble'
]:
    scenario_mask = TRAIN_SCENARIOS[scenario_name]
    scenario_df = model_df.loc[scenario_mask]

    for weighted in (False, True):
        weight_name = (
            'weighted' if weighted else 'unweighted'
        )
        experiment_name = '{}_{}'.format(
            scenario_name,
            weight_name
        )

        (
            selected_params,
            tuning_result,
            performance_history,
            audit_row
        ) = run_anti_overfitting_tuning(
            scenario_df=scenario_df,
            experiment_name=experiment_name,
            weighted=weighted,
            feature_cols=feature_cols,
            best_weight_params=best_weight_params
        )

        tuned_params_by_experiment[
            experiment_name
        ] = selected_params

        tuner_results_by_experiment[
            experiment_name
        ] = tuning_result

        tuner_history_parts.append(
            performance_history
        )
        tuner_audit_rows.append(audit_row)

    del scenario_df
    gc.collect()

tuner_audit = pd.DataFrame(tuner_audit_rows)
tuner_performance_history = pd.concat(
    tuner_history_parts,
    ignore_index=True
)

if len(tuned_params_by_experiment) != 4:
    raise AssertionError(
        'S2/S4 应生成 4 组调参结果，实际为 {}'.format(
            len(tuned_params_by_experiment)
        )
    )

if tuner_audit['oos_overlap_count'].sum() != 0:
    raise AssertionError(
        'AntiOverfittingTuner 调参数据包含 3-oos'
    )

display(tuner_audit)
display(
    tuner_performance_history.sort_values(
        ['experiment', 'final_score'],
        ascending=[True, False]
    )
)


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

,experiment,weighted,tune_n,tune_bad_count,tune_bad_rate,oos_overlap_count,objective,n_trials,cv_folds,best_score_with_penalty,recommended_val_score,valid_trials,overfitting_rate,best_valid_auc_gap,best_valid_ks_gap,best_params,recommended_params
0,S2_train_plus_ensemble_zhongyin_unweighted,False,1095017,53755,0.049091,0,AUCObjective,30,5,0.680983,0.680983,14,0.533333,0.020728,0.028461,"{'boosting_type': 'goss', 'max_depth': 4, 'num...","{'boosting_type': 'goss', 'num_leaves': 15, 'm..."
1,S2_train_plus_ensemble_zhongyin_weighted,True,1095017,53755,0.049091,0,AUCObjective,30,5,0.592765,0.592765,20,0.333333,0.016743,0.022075,"{'boosting_type': 'gbdt', 'max_depth': 5, 'num...","{'boosting_type': 'gbdt', 'num_leaves': 31, 'm..."
2,S4_train_plus_all_ensemble_unweighted,False,1555645,97094,0.062414,0,AUCObjective,30,5,0.671362,0.671362,22,0.266667,0.022114,0.029683,"{'boosting_type': 'goss', 'max_depth': 4, 'num...","{'boosting_type': 'goss', 'num_leaves': 15, 'm..."
3,S4_train_plus_all_ensemble_weighted,True,1555645,97094,0.062414,0,AUCObjective,30,5,0.609333,0.609333,26,0.133333,0.001120,0.000537,"{'boosting_type': 'dart', 'max_depth': 6, 'num...","{'boosting_type': 'dart', 'num_leaves': 63, 'm..."


,experiment,trial,params,train_score,val_score,final_score,train_auc,val_auc,train_ks,val_ks,auc_gap,ks_gap,is_overfitting,total_penalty,train_score_std,val_score_std,train_auc_std,val_auc_std
1,S2_train_plus_ensemble_zhongyin_unweighted,1,"{'boosting_type': 'goss', 'num_leaves': 15, 'm...",0.701711,0.680983,0.680983,0.701711,0.680983,0.295357,0.266896,0.020728,0.028461,False,0.000000,0.000597,0.003878,0.000597,0.003878
7,S2_train_plus_ensemble_zhongyin_unweighted,7,"{'boosting_type': 'dart', 'num_leaves': 13, 'm...",0.697094,0.680350,0.680350,0.697094,0.680350,0.289866,0.266790,0.016744,0.023076,False,0.000000,0.001559,0.003777,0.001559,0.003777
11,S2_train_plus_ensemble_zhongyin_unweighted,11,"{'boosting_type': 'goss', 'num_leaves': 7, 'ma...",0.690509,0.676907,0.676907,0.690509,0.676907,0.278871,0.260198,0.013602,0.018673,False,0.000000,0.000770,0.003728,0.000770,0.003728
23,S2_train_plus_ensemble_zhongyin_unweighted,23,"{'boosting_type': 'goss', 'num_leaves': 15, 'm...",0.707314,0.682201,0.673933,0.707314,0.682201,0.303804,0.269671,0.025114,0.034134,True,0.008267,0.001025,0.003256,0.001025,0.003256
21,S2_train_plus_ensemble_zhongyin_unweighted,21,"{'boosting_type': 'goss', 'num_leaves': 7, 'ma...",0.685127,0.673547,0.673547,0.685127,0.673547,0.270543,0.255436,0.011580,0.015107,False,0.000000,0.000616,0.003735,0.000616,0.003735
25,S2_train_plus_ensemble_zhongyin_unweighted,25,"{'boosting_type': 'dart', 'num_leaves': 15, 'm...",0.713454,0.686210,0.672464,0.713454,0.686210,0.313288,0.276415,0.027244,0.036873,True,0.013746,0.000558,0.003743,0.000558,0.003743
22,S2_train_plus_ensemble_zhongyin_unweighted,22,"{'boosting_type': 'goss', 'num_leaves': 7, 'ma...",0.681790,0.672092,0.672092,0.681790,0.672092,0.265732,0.252848,0.009698,0.012884,False,0.000000,0.000557,0.004377,0.000557,0.004377
8,S2_train_plus_ensemble_zhongyin_unweighted,8,"{'boosting_type': 'dart', 'num_leaves': 14, 'm...",0.684568,0.672065,0.672065,0.684568,0.672065,0.270239,0.253968,0.012503,0.016271,False,0.000000,0.000773,0.004551,0.000773,0.004551
24,S2_train_plus_ensemble_zhongyin_unweighted,24,"{'boosting_type': 'goss', 'num_leaves': 7, 'ma...",0.675511,0.667748,0.667748,0.675511,0.667748,0.255912,0.245598,0.007764,0.010314,False,0.000000,0.000844,0.004239,0.000844,0.004239
2,S2_train_plus_ensemble_zhongyin_unweighted,2,"{'boosting_type': 'goss', 'num_leaves': 15, 'm...",0.676358,0.666339,0.666339,0.676358,0.666339,0.257012,0.243447,0.010019,0.013564,False,0.000000,0.000868,0.004383,0.000868,0.004383


In [65]:
TUNED_SCENARIOS = {
    'S2_train_plus_ensemble_zhongyin',
    'S4_train_plus_all_ensemble'
}

In [66]:
def fit_experiment_lgbm(
    train_df,
    model_params,
    sample_weight=None
):
    if train_df.empty:
        raise ValueError('训练集不能为空')

    if train_df[LABEL_COL].isna().any():
        raise ValueError('训练标签存在空值')

    validate_binary_values(
        train_df[LABEL_COL],
        'train.flag',
        require_both=True
    )

    params = dict(model_params)
    model = lgb.LGBMClassifier(**params)

    model.fit(
        train_df[feature_cols],
        train_df[LABEL_COL].astype(int),
        sample_weight=sample_weight
    )

    return model

prediction_columns = [
    ID_COL,
    LABEL_COL,
    PARTNER_COL,
    PARTNER_CODE_COL,
    SAMPLE_COL,
    BASE_COL
]

# 每个实验只保存其实际训练范围内的预测。
prediction_df = model_df.loc[
    labeled_mask,
    prediction_columns
].copy()

experiment_rows = []
experiment_order = []

for scenario_name, scenario_mask in TRAIN_SCENARIOS.items():
    scenario_df = model_df.loc[scenario_mask]

    if scenario_df[SAMPLE_COL].eq('3-oos').any():
        raise AssertionError(
            '{} 训练范围包含 3-oos'.format(
                scenario_name
            )
        )

    for weighted in (False, True):
        weight_name = (
            'weighted' if weighted else 'unweighted'
        )
        experiment_name = '{}_{}'.format(
            scenario_name,
            weight_name
        )
        experiment_order.append(experiment_name)

        sample_weight = None
        if weighted:
            sample_weight = balance_df_weight(
                df=scenario_df,
                gp_col='weight_group',
                label_col=LABEL_COL,
                base_cnt=best_weight_params['base_cnt'],
                base_rate=best_weight_params['base_rate']
            )

        if scenario_name in TUNED_SCENARIOS:
            model_params = dict(
                tuned_params_by_experiment[
                    experiment_name
                ]
            )
            parameter_source = (
                'AntiOverfittingTuner.recommended_params'
            )
        else:
            model_params = dict(LGB_PARAMS)
            parameter_source = 'fixed_LGB_PARAMS'

        experiment_model = fit_experiment_lgbm(
            train_df=scenario_df,
            model_params=model_params,
            sample_weight=sample_weight
        )

        train_pred = experiment_model.predict_proba(
            scenario_df[feature_cols]
        )[:, 1]

        if not np.isfinite(train_pred).all():
            raise ValueError(
                '{} 训练集预测包含 NaN/inf'.format(
                    experiment_name
                )
            )

        prediction_df[experiment_name] = np.nan
        prediction_df.loc[
            scenario_df.index,
            experiment_name
        ] = train_pred

        experiment_rows.append({
            'experiment': experiment_name,
            'scenario': scenario_name,
            'weighted': weighted,
            'parameter_source': parameter_source,
            'train_n': int(len(scenario_df)),
            'train_bad_count': int(
                scenario_df[LABEL_COL].sum()
            ),
            'train_bad_rate': float(
                scenario_df[LABEL_COL].mean()
            ),
            'base_cnt': (
                best_weight_params['base_cnt']
                if weighted else np.nan
            ),
            'base_rate': (
                best_weight_params['base_rate']
                if weighted else np.nan
            ),
            'model_params': model_params
        })

        del (
            experiment_model,
            train_pred,
            sample_weight
        )
        gc.collect()

    del scenario_df
    gc.collect()

experiment_registry = pd.DataFrame(
    experiment_rows
)

if (
    len(experiment_registry) != 8
    or experiment_registry['experiment'].nunique() != 8
):
    raise AssertionError(
        '实验数量必须严格等于 8'
    )

display(experiment_registry)

,experiment,scenario,weighted,parameter_source,train_n,train_bad_count,train_bad_rate,base_cnt,base_rate,model_params
0,S1_train_only_unweighted,S1_train_only,False,fixed_LGB_PARAMS,735017,18755,0.025516,NaN,NaN,"{'objective': 'binary', 'metric': 'auc', 'boos..."
1,S1_train_only_weighted,S1_train_only,True,fixed_LGB_PARAMS,735017,18755,0.025516,30000.0,0.04,"{'objective': 'binary', 'metric': 'auc', 'boos..."
2,S2_train_plus_ensemble_zhongyin_unweighted,S2_train_plus_ensemble_zhongyin,False,AntiOverfittingTuner.recommended_params,1095017,53755,0.049091,NaN,NaN,"{'boosting_type': 'goss', 'num_leaves': 15, 'm..."
3,S2_train_plus_ensemble_zhongyin_weighted,S2_train_plus_ensemble_zhongyin,True,AntiOverfittingTuner.recommended_params,1095017,53755,0.049091,30000.0,0.04,"{'boosting_type': 'gbdt', 'num_leaves': 31, 'm..."
4,S3_train_plus_ensemble_non_zhongyin_unweighted,S3_train_plus_ensemble_non_zhongyin,False,fixed_LGB_PARAMS,1195645,62094,0.051933,NaN,NaN,"{'objective': 'binary', 'metric': 'auc', 'boos..."
5,S3_train_plus_ensemble_non_zhongyin_weighted,S3_train_plus_ensemble_non_zhongyin,True,fixed_LGB_PARAMS,1195645,62094,0.051933,30000.0,0.04,"{'objective': 'binary', 'metric': 'auc', 'boos..."
6,S4_train_plus_all_ensemble_unweighted,S4_train_plus_all_ensemble,False,AntiOverfittingTuner.recommended_params,1555645,97094,0.062414,NaN,NaN,"{'boosting_type': 'goss', 'num_leaves': 15, 'm..."
7,S4_train_plus_all_ensemble_weighted,S4_train_plus_all_ensemble,True,AntiOverfittingTuner.recommended_params,1555645,97094,0.062414,30000.0,0.04,"{'boosting_type': 'dart', 'num_leaves': 63, 'm..."


In [67]:

def compare_model_with_base(
    eval_df,
    pred_col,
    scenario_name,
    scope_type,
    scope_value
):
    labeled_count = int(
        eval_df[LABEL_COL].notna().sum()
    )

    base_score = pd.to_numeric(
        eval_df[BASE_COL],
        errors='coerce'
    )
    model_score = pd.to_numeric(
        eval_df[pred_col],
        errors='coerce'
    )

    common_mask = (
        eval_df[LABEL_COL].notna()
        & base_score.notna()
        & model_score.notna()
        & np.isfinite(base_score)
        & np.isfinite(model_score)
    )

    common_df = eval_df.loc[common_mask]

    model_metrics = binary_score_metrics(
        common_df[LABEL_COL],
        common_df[pred_col]
    )
    base_metrics = binary_score_metrics(
        common_df[LABEL_COL],
        common_df[BASE_COL]
    )

    n = int(len(common_df))
    bad_count = (
        int(common_df[LABEL_COL].sum())
        if n else 0
    )

    return {
        'scenario': scenario_name,
        'scope_type': scope_type,
        'scope_value': scope_value,
        'experiment': pred_col,
        'partner_count': int(
            eval_df[PARTNER_COL].nunique(
                dropna=False
            )
        ),
        'n': n,
        'bad_count': bad_count,
        'bad_rate': (
            float(common_df[LABEL_COL].mean())
            if n else np.nan
        ),
        'coverage': (
            n / labeled_count
            if labeled_count else np.nan
        ),
        'model_auc': model_metrics['auc'],
        'model_ks': model_metrics['ks'],
        'model_score_direction':
            model_metrics['score_direction'],
        'base_auc': base_metrics['auc'],
        'base_ks': base_metrics['ks'],
        'base_score_direction':
            base_metrics['score_direction'],
        'delta_auc': (
            model_metrics['auc']
            - base_metrics['auc']
        ),
        'delta_ks': (
            model_metrics['ks']
            - base_metrics['ks']
        )
    }

MACRO_METRIC_COLS = [
    'model_auc',
    'model_ks',
    'base_auc',
    'base_ks'
]

def build_partner_macro_mean(
    partner_rows,
    scenario_name,
    experiment_name,
    overall_row
):
    partner_result = pd.DataFrame(
        partner_rows
    )

    if partner_result.empty:
        raise ValueError(
            '{} 没有 partner_name 结果'.format(
                experiment_name
            )
        )

    macro_metrics = (
        partner_result[MACRO_METRIC_COLS]
        .mean(skipna=True)
        .to_dict()
    )

    mean_row = {
        'scenario': scenario_name,
        'scope_type': 'mean',
        'scope_value': 'partner_macro_mean',
        'experiment': experiment_name,
        'partner_count': int(
            len(partner_result)
        ),

        # 样本统计使用实际训练集整体值。
        'n': overall_row['n'],
        'bad_count': overall_row['bad_count'],
        'bad_rate': overall_row['bad_rate'],
        'coverage': overall_row['coverage'],

        # AUC/KS 为各 partner_name 的宏平均。
        'model_auc':
            macro_metrics['model_auc'],
        'model_ks':
            macro_metrics['model_ks'],
        'base_auc':
            macro_metrics['base_auc'],
        'base_ks':
            macro_metrics['base_ks'],

        'delta_auc': (
            macro_metrics['model_auc']
            - macro_metrics['base_auc']
        ),
        'delta_ks': (
            macro_metrics['model_ks']
            - macro_metrics['base_ks']
        ),

        'model_score_direction':
            'partner_macro_mean',
        'base_score_direction':
            'partner_macro_mean'
    }

    return mean_row

result_rows = []

for registry_row in experiment_registry.itertuples(
    index=False
):
    experiment_name = registry_row.experiment
    scenario_name = registry_row.scenario

    scenario_mask = TRAIN_SCENARIOS[
        scenario_name
    ].reindex(
        prediction_df.index,
        fill_value=False
    )

    actual_train_df = prediction_df.loc[
        scenario_mask
    ].copy()

    overall_row = compare_model_with_base(
        eval_df=actual_train_df,
        pred_col=experiment_name,
        scenario_name=scenario_name,
        scope_type='overall',
        scope_value='actual_train'
    )
    result_rows.append(overall_row)

    partner_rows = []

    for partner_name, partner_df in (
        actual_train_df.groupby(
            PARTNER_COL,
            dropna=False,
            sort=True
        )
    ):
        partner_row = compare_model_with_base(
            eval_df=partner_df,
            pred_col=experiment_name,
            scenario_name=scenario_name,
            scope_type='partner_name',
            scope_value=str(partner_name)
        )

        partner_rows.append(partner_row)
        result_rows.append(partner_row)

    mean_row = build_partner_macro_mean(
        partner_rows=partner_rows,
        scenario_name=scenario_name,
        experiment_name=experiment_name,
        overall_row=overall_row
    )
    result_rows.append(mean_row)

In [72]:
oos_prediction_df.head()

,test_id,flag,partner_name,partner_code,mid_sample_type,ym_proba_a077_3,S1_train_only_unweighted,S1_train_only_weighted,S2_train_plus_ensemble_zhongyin_unweighted,S2_train_plus_ensemble_zhongyin_weighted,S3_train_plus_ensemble_non_zhongyin_unweighted,S3_train_plus_ensemble_non_zhongyin_weighted,S4_train_plus_all_ensemble_unweighted,S4_train_plus_all_ensemble_weighted
861159,4491235,0.0,lexin_type2,c_19,3-oos,0.015791,0.017216,0.018238,0.023072,0.018793,0.049208,0.014765,0.046420,0.014420
861160,4491238,0.0,lexin_type2,c_19,3-oos,0.022935,0.020459,0.023324,0.036012,0.027657,0.062692,0.026999,0.064484,0.027680
861161,4491240,0.0,lexin_type2,c_19,3-oos,0.030106,0.029071,0.038350,0.043219,0.037702,0.046265,0.037241,0.058370,0.038571
861162,4491242,0.0,lexin_type2,c_19,3-oos,0.050666,0.031867,0.046754,0.065616,0.047111,0.065718,0.046410,0.078471,0.047358
861163,4491248,0.0,lexin_type2,c_19,3-oos,0.040962,0.034504,0.066152,0.054586,0.057329,0.053201,0.060547,0.070170,0.061524


In [68]:

result_long = pd.DataFrame(result_rows)

scope_order = {
    'overall': 0,
    'mean': 1,
    'partner_name': 2
}
experiment_order_map = {
    experiment_name: order
    for order, experiment_name
    in enumerate(experiment_order)
}

result_long['_scope_order'] = (
    result_long['scope_type']
    .map(scope_order)
)
result_long['_experiment_order'] = (
    result_long['experiment']
    .map(experiment_order_map)
)

result_long = (
    result_long
    .sort_values([
        '_scope_order',
        'scope_value',
        '_experiment_order'
    ])
    .drop(columns=[
        '_scope_order',
        '_experiment_order'
    ])
    .reset_index(drop=True)
)

# ============================================================
# 7. result_wide
# ============================================================

WIDE_METRIC_COLS = [
    'partner_count',
    'n',
    'bad_count',
    'bad_rate',
    'coverage',
    'model_auc',
    'base_auc',
    'delta_auc',
    'model_ks',
    'base_ks',
    'delta_ks'
]

result_wide = result_long.pivot(
    index=[
        'scope_type',
        'scope_value'
    ],
    columns='experiment',
    values=WIDE_METRIC_COLS
)

expected_wide_columns = pd.MultiIndex.from_product(
    [
        WIDE_METRIC_COLS,
        experiment_order
    ],
    names=[
        'metric',
        'experiment'
    ]
)

result_wide = (
    result_wide
    .reindex(columns=expected_wide_columns)
    .reset_index()
)

In [70]:
result_wide

metric        scope_type             scope_value            partner_count  \
experiment                                       S1_train_only_unweighted   
0                   mean      partner_macro_mean                      6.0   
1                overall            actual_train                      6.0   
2           partner_name             lexin_type5                      1.0   
3           partner_name  weipingfubang_lowerirr                      NaN   
4           partner_name                      中银                      NaN   
5           partner_name                  海尔(借呗)                      1.0   
6           partner_name            海尔(借呗拒绝其他通过)                      1.0   
7           partner_name                海尔(字节星选)                      1.0   
8           partner_name          海尔(字节星选拒绝其他通过)                      1.0   
9           partner_name                海尔(字节智选)                      1.0   
10          partner_name                海尔(自营贷中)                      NaN   
11          partner_name                  海尔(蚂蚁)                      NaN   

metric                                                                        \
experiment S1_train_only_weighted S2_train_plus_ensemble_zhongyin_unweighted   
0                             6.0                                        7.0   
1                             6.0                                        7.0   
2                             1.0                                        1.0   
3                             NaN                                        NaN   
4                             NaN                                        1.0   
5                             1.0                                        1.0   
6                             1.0                                        1.0   
7                             1.0                                        1.0   
8                             1.0                                        1.0   
9                             1.0                                        1.0   
10                            NaN                                        NaN   
11                            NaN                                        NaN   

metric                                               \
experiment S2_train_plus_ensemble_zhongyin_weighted   
0                                               7.0   
1                                               7.0   
2                                               1.0   
3                                               NaN   
4                                               1.0   
5                                               1.0   
6                                               1.0   
7                                               1.0   
8                                               1.0   
9                                               1.0   
10                                              NaN   
11                                              NaN   

metric                                                     \
experiment S3_train_plus_ensemble_non_zhongyin_unweighted   
0                                                     9.0   
1                                                     9.0   
2                                                     1.0   
3                                                     1.0   
4                                                     NaN   
5                                                     1.0   
6                                                     1.0   
7                                                     1.0   
8                                                     1.0   
9                                                     1.0   
10                                                    1.0   
11                                                    1.0   

metric                                                   \
experiment S3_train_plus_ensemble_non_zhongyin_weighted   
0                                                   9.0 

In [71]:
result_wide.to_excel('2-train_wide.xlsx')

In [69]:

result_long_summary = (
    result_long.loc[
        result_long['scope_type'].isin([
            'overall',
            'mean'
        ])
    ]
    .reset_index(drop=True)
)

summary_index = result_wide[
    'scope_type'
].isin([
    'overall',
    'mean'
])

result_wide_summary = (
    result_wide.loc[summary_index]
    .reset_index(drop=True)
)

display(result_long)
display(result_wide)

display(result_long_summary)
display(result_wide_summary)

,scenario,scope_type,scope_value,experiment,partner_count,n,bad_count,bad_rate,coverage,model_auc,model_ks,model_score_direction,base_auc,base_ks,base_score_direction,delta_auc,delta_ks
0,S1_train_only,overall,actual_train,S1_train_only_unweighted,6,735017,18755,0.025516,1.000000,0.654063,0.216468,higher_is_bad,0.637756,0.194369,higher_is_bad,0.016307,0.022099
1,S1_train_only,overall,actual_train,S1_train_only_weighted,6,735017,18755,0.025516,1.000000,0.631754,0.186643,higher_is_bad,0.637756,0.194369,higher_is_bad,-0.006003,-0.007725
2,S2_train_plus_ensemble_zhongyin,overall,actual_train,S2_train_plus_ensemble_zhongyin_unweighted,7,1084406,52646,0.048548,0.990310,0.698049,0.290254,higher_is_bad,0.603620,0.148010,higher_is_bad,0.094429,0.142244
3,S2_train_plus_ensemble_zhongyin,overall,actual_train,S2_train_plus_ensemble_zhongyin_weighted,7,1084406,52646,0.048548,0.990310,0.607830,0.153845,higher_is_bad,0.603620,0.148010,higher_is_bad,0.004209,0.005835
4,S3_train_plus_ensemble_non_zhongyin,overall,actual_train,S3_train_plus_ensemble_non_zhongyin_unweighted,9,1195645,62094,0.051933,1.000000,0.695360,0.290116,higher_is_bad,0.661293,0.234421,higher_is_bad,0.034066,0.055696
5,S3_train_plus_ensemble_non_zhongyin,overall,actual_train,S3_train_plus_ensemble_non_zhongyin_weighted,9,1195645,62094,0.051933,1.000000,0.629863,0.184640,higher_is_bad,0.661293,0.234421,higher_is_bad,-0.031430,-0.049780
6,S4_train_plus_all_ensemble,overall,actual_train,S4_train_plus_all_ensemble_unweighted,10,1545034,95985,0.062125,0.993179,0.690512,0.276586,higher_is_bad,0.631914,0.190252,higher_is_bad,0.058598,0.086334
7,S4_train_plus_all_ensemble,overall,actual_train,S4_train_plus_all_ensemble_weighted,10,1545034,95985,0.062125,0.993179,0.610417,0.165392,higher_is_bad,0.631914,0.190252,higher_is_bad,-0.021497,-0.024860
8,S1_train_only,mean,partner_macro_mean,S1_train_only_unweighted,6,735017,18755,0.025516,1.000000,0.656897,0.230328,partner_macro_mean,0.636757,0.200408,partner_macro_mean,0.020140,0.029920
9,S1_train_only,mean,partner_macro_mean,S1_train_only_weighted,6,735017,18755,0.025516,1.000000,0.703682,0.305485,partner_macro_mean,0.636757,0.200408,partner_macro_mean,0.066925,0.105077


metric        scope_type             scope_value            partner_count  \
experiment                                       S1_train_only_unweighted   
0                   mean      partner_macro_mean                      6.0   
1                overall            actual_train                      6.0   
2           partner_name             lexin_type5                      1.0   
3           partner_name  weipingfubang_lowerirr                      NaN   
4           partner_name                      中银                      NaN   
5           partner_name                  海尔(借呗)                      1.0   
6           partner_name            海尔(借呗拒绝其他通过)                      1.0   
7           partner_name                海尔(字节星选)                      1.0   
8           partner_name          海尔(字节星选拒绝其他通过)                      1.0   
9           partner_name                海尔(字节智选)                      1.0   
10          partner_name                海尔(自营贷中)                      NaN   
11          partner_name                  海尔(蚂蚁)                      NaN   

metric                                                                        \
experiment S1_train_only_weighted S2_train_plus_ensemble_zhongyin_unweighted   
0                             6.0                                        7.0   
1                             6.0                                        7.0   
2                             1.0                                        1.0   
3                             NaN                                        NaN   
4                             NaN                                        1.0   
5                             1.0                                        1.0   
6                             1.0                                        1.0   
7                             1.0                                        1.0   
8                             1.0                                        1.0   
9                             1.0                                        1.0   
10                            NaN                                        NaN   
11                            NaN                                        NaN   

metric                                               \
experiment S2_train_plus_ensemble_zhongyin_weighted   
0                                               7.0   
1                                               7.0   
2                                               1.0   
3                                               NaN   
4                                               1.0   
5                                               1.0   
6                                               1.0   
7                                               1.0   
8                                               1.0   
9                                               1.0   
10                                              NaN   
11                                              NaN   

metric                                                     \
experiment S3_train_plus_ensemble_non_zhongyin_unweighted   
0                                                     9.0   
1                                                     9.0   
2                                                     1.0   
3                                                     1.0   
4                                                     NaN   
5                                                     1.0   
6                                                     1.0   
7                                                     1.0   
8                                                     1.0   
9                                                     1.0   
10                                                    1.0   
11                                                    1.0   

metric                                                   \
experiment S3_train_plus_ensemble_non_zhongyin_weighted   
0                                                   9.0 

,scenario,scope_type,scope_value,experiment,partner_count,n,bad_count,bad_rate,coverage,model_auc,model_ks,model_score_direction,base_auc,base_ks,base_score_direction,delta_auc,delta_ks
0,S1_train_only,overall,actual_train,S1_train_only_unweighted,6,735017,18755,0.025516,1.000000,0.654063,0.216468,higher_is_bad,0.637756,0.194369,higher_is_bad,0.016307,0.022099
1,S1_train_only,overall,actual_train,S1_train_only_weighted,6,735017,18755,0.025516,1.000000,0.631754,0.186643,higher_is_bad,0.637756,0.194369,higher_is_bad,-0.006003,-0.007725
2,S2_train_plus_ensemble_zhongyin,overall,actual_train,S2_train_plus_ensemble_zhongyin_unweighted,7,1084406,52646,0.048548,0.990310,0.698049,0.290254,higher_is_bad,0.603620,0.148010,higher_is_bad,0.094429,0.142244
3,S2_train_plus_ensemble_zhongyin,overall,actual_train,S2_train_plus_ensemble_zhongyin_weighted,7,1084406,52646,0.048548,0.990310,0.607830,0.153845,higher_is_bad,0.603620,0.148010,higher_is_bad,0.004209,0.005835
4,S3_train_plus_ensemble_non_zhongyin,overall,actual_train,S3_train_plus_ensemble_non_zhongyin_unweighted,9,1195645,62094,0.051933,1.000000,0.695360,0.290116,higher_is_bad,0.661293,0.234421,higher_is_bad,0.034066,0.055696
5,S3_train_plus_ensemble_non_zhongyin,overall,actual_train,S3_train_plus_ensemble_non_zhongyin_weighted,9,1195645,62094,0.051933,1.000000,0.629863,0.184640,higher_is_bad,0.661293,0.234421,higher_is_bad,-0.031430,-0.049780
6,S4_train_plus_all_ensemble,overall,actual_train,S4_train_plus_all_ensemble_unweighted,10,1545034,95985,0.062125,0.993179,0.690512,0.276586,higher_is_bad,0.631914,0.190252,higher_is_bad,0.058598,0.086334
7,S4_train_plus_all_ensemble,overall,actual_train,S4_train_plus_all_ensemble_weighted,10,1545034,95985,0.062125,0.993179,0.610417,0.165392,higher_is_bad,0.631914,0.190252,higher_is_bad,-0.021497,-0.024860
8,S1_train_only,mean,partner_macro_mean,S1_train_only_unweighted,6,735017,18755,0.025516,1.000000,0.656897,0.230328,partner_macro_mean,0.636757,0.200408,partner_macro_mean,0.020140,0.029920
9,S1_train_only,mean,partner_macro_mean,S1_train_only_weighted,6,735017,18755,0.025516,1.000000,0.703682,0.305485,partner_macro_mean,0.636757,0.200408,partner_macro_mean,0.066925,0.105077


metric     scope_type         scope_value            partner_count  \
experiment                                S1_train_only_unweighted   
0                mean  partner_macro_mean                      6.0   
1             overall        actual_train                      6.0   

metric                                                                        \
experiment S1_train_only_weighted S2_train_plus_ensemble_zhongyin_unweighted   
0                             6.0                                        7.0   
1                             6.0                                        7.0   

metric                                               \
experiment S2_train_plus_ensemble_zhongyin_weighted   
0                                               7.0   
1                                               7.0   

metric                                                     \
experiment S3_train_plus_ensemble_non_zhongyin_unweighted   
0                                                     9.0   
1                                                     9.0   

metric                                                   \
experiment S3_train_plus_ensemble_non_zhongyin_weighted   
0                                                   9.0   
1                                                   9.0   

metric                                            \
experiment S4_train_plus_all_ensemble_unweighted   
0                                           10.0   
1                                           10.0   

metric                                                                n  \
experiment S4_train_plus_all_ensemble_weighted S1_train_only_unweighted   
0                                         10.0                 735017.0   
1                                         10.0                 735017.0   

metric                                                                        \
experiment S1_train_only_weighted S2_train_plus_ensemble_zhongyin_unweighted   
0                        735017.0                                  1084406.0   
1                        735017.0                                  1084406.0   

metric                                               \
experiment S2_train_plus_ensemble_zhongyin_weighted   
0                                         1084406.0   
1                                         1084406.0   

metric                                                     \
experiment S3_train_plus_ensemble_non_zhongyin_unweighted   
0                                               1195645.0   
1                                               1195645.0   

metric                                                   \
experiment S3_train_plus_ensemble_non_zhongyin_weighted   
0                                             1195645.0   
1                                             1195645.0   

metric                                            \
experiment S4_train_plus_all_ensemble_unweighted   
0                                      1545034.0   
1                                      1545034.0   

metric                                                        bad_count  \
experiment S4_train_plus_all_ensemble_weighted S1_train_only_unweighted   
0                                    1545034.0                  18755.0   
1                                    1545034.0                  18755.0   

metric                                                                        \
experiment S1_train_only_weighted S2_train_plus_ensemble_zhongyin_unweighted   
0                         18755.0                                    52646.0   
1                         18755.0                                    52646.0   

metric                                               \
experiment S2_train_plus_ensemble_zhongyin_weighted   
0                                           52646.0   
1                                           52646.0   

metric                                                     \
experiment S3_train_plus_ensemble_non_zhongyin_u

In [ ]:
result_long.to_excel()

In [75]:
TRAIN_SCENARIOS

{'S1_train_only': 861159     False
 861160     False
 861161     False
 861162     False
 861163     False
            ...  
 4353499    False
 4353500    False
 4353501    False
 4353502    False
 4353503    False
 Length: 3181655, dtype: bool,
 'S2_train_plus_ensemble_zhongyin': 861159     False
 861160     False
 861161     False
 861162     False
 861163     False
            ...  
 4353499     True
 4353500     True
 4353501     True
 4353502     True
 4353503     True
 Length: 3181655, dtype: bool,
 'S3_train_plus_ensemble_non_zhongyin': 861159     False
 861160     False
 861161     False
 861162     False
 861163     False
            ...  
 4353499    False
 4353500    False
 4353501    False
 4353502    False
 4353503    False
 Length: 3181655, dtype: bool,
 'S4_train_plus_all_ensemble': 861159     False
 861160     False
 861161     False
 861162     False
 861163     False
            ...  
 4353499     True
 4353500     True
 4353501     True
 4353502     True
 4353503    

In [76]:
# experiment_model 只在训练循环内部存在。
# 请运行下面整个 cell，不要单独运行 predict_proba 片段。

def fit_experiment_lgbm(
    train_df,
    model_params,
    sample_weight=None
):
    if train_df.empty:
        raise ValueError('训练集不能为空')

    validate_binary_values(
        train_df[LABEL_COL],
        'train.flag',
        require_both=True
    )

    model = lgb.LGBMClassifier(
        **dict(model_params)
    )

    model.fit(
        train_df[feature_cols],
        train_df[LABEL_COL].astype(int),
        sample_weight=sample_weight
    )

    return model

labeled_mask = model_df[LABEL_COL].notna()
oos_mask = (
    model_df[SAMPLE_COL].eq('3-oos')
    & labeled_mask
)

prediction_columns = [
    ID_COL,
    LABEL_COL,
    PARTNER_COL,
    PARTNER_CODE_COL,
    SAMPLE_COL,
    BASE_COL
]

prediction_df = model_df.loc[
    labeled_mask,
    prediction_columns
].copy()

experiment_rows = []
experiment_order = []

for scenario_name, scenario_mask in TRAIN_SCENARIOS.items():
    scenario_df = model_df.loc[
        scenario_mask
    ]

    if scenario_df[SAMPLE_COL].eq('3-oos').any():
        raise AssertionError(
            '{} 的训练范围包含 3-oos'.format(
                scenario_name
            )
        )

    for weighted in (False, True):
        weight_name = (
            'weighted'
            if weighted
            else 'unweighted'
        )

        experiment_name = '{}_{}'.format(
            scenario_name,
            weight_name
        )
        experiment_order.append(
            experiment_name
        )

        # 计算当前实验训练权重
        sample_weight = None

        if weighted:
            sample_weight = balance_df_weight(
                df=scenario_df,
                gp_col='weight_group',
                label_col=LABEL_COL,
                base_cnt=best_weight_params[
                    'base_cnt'
                ],
                base_rate=best_weight_params[
                    'base_rate'
                ]
            )

        # S2/S4 使用 AntiOverfittingTuner 参数；
        # S1/S3 使用固定参数。
        if scenario_name in TUNED_SCENARIOS:
            if (
                experiment_name
                not in tuned_params_by_experiment
            ):
                raise KeyError(
                    '{} 缺少调参结果'.format(
                        experiment_name
                    )
                )

            model_params = dict(
                tuned_params_by_experiment[
                    experiment_name
                ]
            )
            parameter_source = (
                'AntiOverfittingTuner.'
                'recommended_params'
            )
        else:
            model_params = dict(
                LGB_PARAMS
            )
            parameter_source = (
                'fixed_LGB_PARAMS'
            )

        # 每个实验必须先重新创建并 fit 模型
        experiment_model = fit_experiment_lgbm(
            train_df=scenario_df,
            model_params=model_params,
            sample_weight=sample_weight
        )

        prediction_df[experiment_name] = np.nan

        # 当前实验实际训练集预测
        actual_train_pred = (
            experiment_model.predict_proba(
                scenario_df[feature_cols]
            )[:, 1]
        )

        if not np.isfinite(
            actual_train_pred
        ).all():
            raise ValueError(
                '{} 的 actual_train 预测包含 '
                'NaN/inf'.format(
                    experiment_name
                )
            )

        prediction_df.loc[
            scenario_df.index,
            experiment_name
        ] = actual_train_pred

        # 当前实验 3-oos 预测
        oos_feature_df = model_df.loc[
            oos_mask,
            feature_cols
        ]

        oos_pred = (
            experiment_model.predict_proba(
                oos_feature_df
            )[:, 1]
        )

        if not np.isfinite(oos_pred).all():
            raise ValueError(
                '{} 的 3-oos 预测包含 '
                'NaN/inf'.format(
                    experiment_name
                )
            )

        prediction_df.loc[
            oos_feature_df.index,
            experiment_name
        ] = oos_pred

        experiment_rows.append({
            'experiment': experiment_name,
            'scenario': scenario_name,
            'weighted': weighted,
            'parameter_source':
                parameter_source,
            'train_n': int(
                len(scenario_df)
            ),
            'train_bad_count': int(
                scenario_df[
                    LABEL_COL
                ].sum()
            ),
            'train_bad_rate': float(
                scenario_df[
                    LABEL_COL
                ].mean()
            ),
            'base_cnt': (
                best_weight_params[
                    'base_cnt'
                ]
                if weighted
                else np.nan
            ),
            'base_rate': (
                best_weight_params[
                    'base_rate'
                ]
                if weighted
                else np.nan
            ),
            'model_params': model_params
        })

        del (
            experiment_model,
            actual_train_pred,
            oos_pred,
            oos_feature_df,
            sample_weight
        )
        gc.collect()

    del scenario_df
    gc.collect()

experiment_registry = pd.DataFrame(
    experiment_rows
)

if (
    len(experiment_registry) != 8
    or experiment_registry[
        'experiment'
    ].nunique() != 8
):
    raise AssertionError(
        '实验数量必须严格等于 8'
    )

# 确认八组实验都有 actual_train 和 OOS 预测
for registry_row in (
    experiment_registry.itertuples(
        index=False
    )
):
    experiment_name = (
        registry_row.experiment
    )
    scenario_name = registry_row.scenario

    actual_mask = (
        TRAIN_SCENARIOS[
            scenario_name
        ].reindex(
            prediction_df.index,
            fill_value=False
        )
    )
    prediction_oos_mask = (
        prediction_df[
            SAMPLE_COL
        ].eq('3-oos')
    )

    if prediction_df.loc[
        actual_mask,
        experiment_name
    ].isna().any():
        raise ValueError(
            '{} 的 actual_train 预测不完整'.format(
                experiment_name
            )
        )

    if prediction_df.loc[
        prediction_oos_mask,
        experiment_name
    ].isna().any():
        raise ValueError(
            '{} 的 3-oos 预测不完整'.format(
                experiment_name
            )
        )

display(experiment_registry)


,experiment,scenario,weighted,parameter_source,train_n,train_bad_count,train_bad_rate,base_cnt,base_rate,model_params
0,S1_train_only_unweighted,S1_train_only,False,fixed_LGB_PARAMS,735017,18755,0.025516,NaN,NaN,"{'objective': 'binary', 'metric': 'auc', 'boos..."
1,S1_train_only_weighted,S1_train_only,True,fixed_LGB_PARAMS,735017,18755,0.025516,30000.0,0.04,"{'objective': 'binary', 'metric': 'auc', 'boos..."
2,S2_train_plus_ensemble_zhongyin_unweighted,S2_train_plus_ensemble_zhongyin,False,AntiOverfittingTuner.recommended_params,1095017,53755,0.049091,NaN,NaN,"{'boosting_type': 'goss', 'num_leaves': 15, 'm..."
3,S2_train_plus_ensemble_zhongyin_weighted,S2_train_plus_ensemble_zhongyin,True,AntiOverfittingTuner.recommended_params,1095017,53755,0.049091,30000.0,0.04,"{'boosting_type': 'gbdt', 'num_leaves': 31, 'm..."
4,S3_train_plus_ensemble_non_zhongyin_unweighted,S3_train_plus_ensemble_non_zhongyin,False,fixed_LGB_PARAMS,1195645,62094,0.051933,NaN,NaN,"{'objective': 'binary', 'metric': 'auc', 'boos..."
5,S3_train_plus_ensemble_non_zhongyin_weighted,S3_train_plus_ensemble_non_zhongyin,True,fixed_LGB_PARAMS,1195645,62094,0.051933,30000.0,0.04,"{'objective': 'binary', 'metric': 'auc', 'boos..."
6,S4_train_plus_all_ensemble_unweighted,S4_train_plus_all_ensemble,False,AntiOverfittingTuner.recommended_params,1555645,97094,0.062414,NaN,NaN,"{'boosting_type': 'goss', 'num_leaves': 15, 'm..."
7,S4_train_plus_all_ensemble_weighted,S4_train_plus_all_ensemble,True,AntiOverfittingTuner.recommended_params,1555645,97094,0.062414,30000.0,0.04,"{'boosting_type': 'dart', 'num_leaves': 63, 'm..."


In [77]:
prediction_df.head()

,test_id,flag,partner_name,partner_code,mid_sample_type,ym_proba_a077_3,S1_train_only_unweighted,S1_train_only_weighted,S2_train_plus_ensemble_zhongyin_unweighted,S2_train_plus_ensemble_zhongyin_weighted,S3_train_plus_ensemble_non_zhongyin_unweighted,S3_train_plus_ensemble_non_zhongyin_weighted,S4_train_plus_all_ensemble_unweighted,S4_train_plus_all_ensemble_weighted
861159,4491235,0.0,lexin_type2,c_19,3-oos,0.015791,0.017216,0.018238,0.019398,0.015298,0.049208,0.014765,0.045713,0.304888
861160,4491238,0.0,lexin_type2,c_19,3-oos,0.022935,0.020459,0.023324,0.032490,0.029692,0.062692,0.026999,0.063407,0.308840
861161,4491240,0.0,lexin_type2,c_19,3-oos,0.030106,0.029071,0.038350,0.042917,0.041099,0.046265,0.037241,0.057282,0.312471
861162,4491242,0.0,lexin_type2,c_19,3-oos,0.050666,0.031867,0.046754,0.064746,0.052465,0.065718,0.046410,0.080495,0.314629
861163,4491248,0.0,lexin_type2,c_19,3-oos,0.040962,0.034504,0.066152,0.058848,0.077485,0.053201,0.060547,0.066731,0.321247


In [79]:

def append_evaluation_rows(
    result_rows,
    eval_df,
    evaluation_set,
    experiment_name,
    scenario_name
):
    if eval_df.empty:
        raise ValueError(
            '{} / {} 的评估数据为空'.format(
                experiment_name,
                evaluation_set
            )
        )

    # overall
    overall_row = compare_model_with_base(
        eval_df=eval_df,
        pred_col=experiment_name,
        scenario_name=scenario_name,
        scope_type='overall',
        scope_value='all'
    )
    overall_row['evaluation_set'] = evaluation_set
    result_rows.append(overall_row)

    # partner_name
    partner_rows = []

    for partner_name, partner_df in eval_df.groupby(
        PARTNER_COL,
        dropna=False,
        sort=True
    ):
        partner_row = compare_model_with_base(
            eval_df=partner_df,
            pred_col=experiment_name,
            scenario_name=scenario_name,
            scope_type='partner_name',
            scope_value=str(partner_name)
        )
        partner_row['evaluation_set'] = evaluation_set

        partner_rows.append(partner_row)
        result_rows.append(partner_row)

    # partner macro mean
    mean_row = build_partner_macro_mean(
        partner_rows=partner_rows,
        scenario_name=scenario_name,
        experiment_name=experiment_name,
        overall_row=overall_row
    )
    mean_row['evaluation_set'] = evaluation_set
    result_rows.append(mean_row)

# ============================================================
# 3. 分别计算八组实验的 actual_train 和 3-oos 指标
# ============================================================

result_rows = []

for registry_row in experiment_registry.itertuples(
    index=False
):
    experiment_name = registry_row.experiment
    scenario_name = registry_row.scenario

    # 当前实验的实际训练范围
    actual_train_mask = TRAIN_SCENARIOS[
        scenario_name
    ].reindex(
        prediction_df.index,
        fill_value=False
    )

    actual_train_df = prediction_df.loc[
        actual_train_mask
    ].copy()

    append_evaluation_rows(
        result_rows=result_rows,
        eval_df=actual_train_df,
        evaluation_set='actual_train',
        experiment_name=experiment_name,
        scenario_name=scenario_name
    )

    # 所有实验统一使用同一批 3-oos
    experiment_oos_df = prediction_df.loc[
        prediction_df[SAMPLE_COL].eq('3-oos')
    ].copy()

    append_evaluation_rows(
        result_rows=result_rows,
        eval_df=experiment_oos_df,
        evaluation_set='3-oos',
        experiment_name=experiment_name,
        scenario_name=scenario_name
    )

# ============================================================
# 4. result_long
# ============================================================

result_long = pd.DataFrame(result_rows)

evaluation_order = {
    'actual_train': 0,
    '3-oos': 1
}
scope_order = {
    'overall': 0,
    'mean': 1,
    'partner_name': 2
}
experiment_order_map = {
    experiment_name: order
    for order, experiment_name
    in enumerate(experiment_order)
}

result_long['_evaluation_order'] = (
    result_long['evaluation_set']
    .map(evaluation_order)
)
result_long['_scope_order'] = (
    result_long['scope_type']
    .map(scope_order)
)
result_long['_experiment_order'] = (
    result_long['experiment']
    .map(experiment_order_map)
)

result_long = (
    result_long
    .sort_values([
        '_evaluation_order',
        '_scope_order',
        'scope_value',
        '_experiment_order'
    ])
    .drop(columns=[
        '_evaluation_order',
        '_scope_order',
        '_experiment_order'
    ])
    .reset_index(drop=True)
)

result_long_actual_train = (
    result_long.loc[
        result_long['evaluation_set'].eq(
            'actual_train'
        )
    ]
    .reset_index(drop=True)
)

result_long_oos = (
    result_long.loc[
        result_long['evaluation_set'].eq(
            '3-oos'
        )
    ]
    .reset_index(drop=True)
)

# ============================================================
# 5. result_wide 生成函数
# ============================================================

WIDE_METRIC_COLS = [
    'partner_count',
    'n',
    'bad_count',
    'bad_rate',
    'coverage',
    'model_auc',
    'base_auc',
    'delta_auc',
    'model_ks',
    'base_ks',
    'delta_ks'
]

def build_result_wide(
    result_long_df,
    experiment_order
):
    if result_long_df.empty:
        return pd.DataFrame()

    result_wide_df = result_long_df.pivot(
        index=[
            'evaluation_set',
            'scope_type',
            'scope_value'
        ],
        columns='experiment',
        values=WIDE_METRIC_COLS
    )

    expected_columns = pd.MultiIndex.from_product(
        [
            WIDE_METRIC_COLS,
            experiment_order
        ],
        names=[
            'metric',
            'experiment'
        ]
    )

    result_wide_df = (
        result_wide_df
        .reindex(columns=expected_columns)
        .reset_index()
    )

    return result_wide_df

# ============================================================
# 6. 分别生成 actual_train / OOS result_wide，再拼接
# ============================================================

result_wide_actual_train = build_result_wide(
    result_long_df=result_long_actual_train,
    experiment_order=experiment_order
)

result_wide_oos = build_result_wide(
    result_long_df=result_long_oos,
    experiment_order=experiment_order
)

result_wide = pd.concat(
    [
        result_wide_actual_train,
        result_wide_oos
    ],
    axis=0,
    ignore_index=True
)

# ============================================================
# 7. 只保留 overall + mean 的汇总结果
# ============================================================

result_long_summary = (
    result_long.loc[
        result_long['scope_type'].isin([
            'overall',
            'mean'
        ])
    ]
    .reset_index(drop=True)
)

result_wide_summary = (
    result_wide.loc[
        result_wide['scope_type'].isin([
            'overall',
            'mean'
        ])
    ]
    .reset_index(drop=True)
)

display(result_long_actual_train)
display(result_long_oos)

display(result_wide_actual_train)
display(result_wide_oos)

# actual_train 与 3-oos 拼接后的最终宽表
display(result_wide)

# 仅 overall + mean
display(result_long_summary)
display(result_wide_summary)

,scenario,scope_type,scope_value,experiment,partner_count,n,bad_count,bad_rate,coverage,model_auc,model_ks,model_score_direction,base_auc,base_ks,base_score_direction,delta_auc,delta_ks,evaluation_set
0,S1_train_only,overall,all,S1_train_only_unweighted,6,735017,18755,0.025516,1.000000,0.654063,0.216468,higher_is_bad,0.637756,0.194369,higher_is_bad,0.016307,0.022099,actual_train
1,S1_train_only,overall,all,S1_train_only_weighted,6,735017,18755,0.025516,1.000000,0.631754,0.186643,higher_is_bad,0.637756,0.194369,higher_is_bad,-0.006003,-0.007725,actual_train
2,S2_train_plus_ensemble_zhongyin,overall,all,S2_train_plus_ensemble_zhongyin_unweighted,7,1084406,52646,0.048548,0.990310,0.698049,0.290254,higher_is_bad,0.603620,0.148010,higher_is_bad,0.094429,0.142244,actual_train
3,S2_train_plus_ensemble_zhongyin,overall,all,S2_train_plus_ensemble_zhongyin_weighted,7,1084406,52646,0.048548,0.990310,0.607830,0.153845,higher_is_bad,0.603620,0.148010,higher_is_bad,0.004209,0.005835,actual_train
4,S3_train_plus_ensemble_non_zhongyin,overall,all,S3_train_plus_ensemble_non_zhongyin_unweighted,9,1195645,62094,0.051933,1.000000,0.695360,0.290116,higher_is_bad,0.661293,0.234421,higher_is_bad,0.034066,0.055696,actual_train
5,S3_train_plus_ensemble_non_zhongyin,overall,all,S3_train_plus_ensemble_non_zhongyin_weighted,9,1195645,62094,0.051933,1.000000,0.629863,0.184640,higher_is_bad,0.661293,0.234421,higher_is_bad,-0.031430,-0.049780,actual_train
6,S4_train_plus_all_ensemble,overall,all,S4_train_plus_all_ensemble_unweighted,10,1545034,95985,0.062125,0.993179,0.690512,0.276586,higher_is_bad,0.631914,0.190252,higher_is_bad,0.058598,0.086334,actual_train
7,S4_train_plus_all_ensemble,overall,all,S4_train_plus_all_ensemble_weighted,10,1545034,95985,0.062125,0.993179,0.610417,0.165392,higher_is_bad,0.631914,0.190252,higher_is_bad,-0.021497,-0.024860,actual_train
8,S1_train_only,mean,partner_macro_mean,S1_train_only_unweighted,6,735017,18755,0.025516,1.000000,0.656897,0.230328,partner_macro_mean,0.636757,0.200408,partner_macro_mean,0.020140,0.029920,actual_train
9,S1_train_only,mean,partner_macro_mean,S1_train_only_weighted,6,735017,18755,0.025516,1.000000,0.703682,0.305485,partner_macro_mean,0.636757,0.200408,partner_macro_mean,0.066925,0.105077,actual_train


,scenario,scope_type,scope_value,experiment,partner_count,n,bad_count,bad_rate,coverage,model_auc,model_ks,model_score_direction,base_auc,base_ks,base_score_direction,delta_auc,delta_ks,evaluation_set
0,S1_train_only,overall,all,S1_train_only_unweighted,8,1626010,93628,0.057581,1.0,0.663185,0.231818,higher_is_bad,0.665086,0.235111,higher_is_bad,-0.001901,-0.003293,3-oos
1,S1_train_only,overall,all,S1_train_only_weighted,8,1626010,93628,0.057581,1.0,0.658676,0.228993,higher_is_bad,0.665086,0.235111,higher_is_bad,-0.006410,-0.006118,3-oos
2,S2_train_plus_ensemble_zhongyin,overall,all,S2_train_plus_ensemble_zhongyin_unweighted,8,1626010,93628,0.057581,1.0,0.634282,0.191134,higher_is_bad,0.665086,0.235111,higher_is_bad,-0.030804,-0.043977,3-oos
3,S2_train_plus_ensemble_zhongyin,overall,all,S2_train_plus_ensemble_zhongyin_weighted,8,1626010,93628,0.057581,1.0,0.655694,0.222495,higher_is_bad,0.665086,0.235111,higher_is_bad,-0.009392,-0.012616,3-oos
4,S3_train_plus_ensemble_non_zhongyin,overall,all,S3_train_plus_ensemble_non_zhongyin_unweighted,8,1626010,93628,0.057581,1.0,0.621087,0.163802,higher_is_bad,0.665086,0.235111,higher_is_bad,-0.043998,-0.071309,3-oos
5,S3_train_plus_ensemble_non_zhongyin,overall,all,S3_train_plus_ensemble_non_zhongyin_weighted,8,1626010,93628,0.057581,1.0,0.664649,0.235902,higher_is_bad,0.665086,0.235111,higher_is_bad,-0.000436,0.000791,3-oos
6,S4_train_plus_all_ensemble,overall,all,S4_train_plus_all_ensemble_unweighted,8,1626010,93628,0.057581,1.0,0.629921,0.184031,higher_is_bad,0.665086,0.235111,higher_is_bad,-0.035164,-0.051080,3-oos
7,S4_train_plus_all_ensemble,overall,all,S4_train_plus_all_ensemble_weighted,8,1626010,93628,0.057581,1.0,0.663095,0.236062,higher_is_bad,0.665086,0.235111,higher_is_bad,-0.001991,0.000951,3-oos
8,S1_train_only,mean,partner_macro_mean,S1_train_only_unweighted,8,1626010,93628,0.057581,1.0,0.628024,0.185307,partner_macro_mean,0.633876,0.193185,partner_macro_mean,-0.005852,-0.007878,3-oos
9,S1_train_only,mean,partner_macro_mean,S1_train_only_weighted,8,1626010,93628,0.057581,1.0,0.612523,0.162661,partner_macro_mean,0.633876,0.193185,partner_macro_mean,-0.021353,-0.030524,3-oos


metric     evaluation_set    scope_type             scope_value  \
experiment                                                        
0            actual_train          mean      partner_macro_mean   
1            actual_train       overall                     all   
2            actual_train  partner_name             lexin_type5   
3            actual_train  partner_name  weipingfubang_lowerirr   
4            actual_train  partner_name                      中银   
5            actual_train  partner_name                  海尔(借呗)   
6            actual_train  partner_name            海尔(借呗拒绝其他通过)   
7            actual_train  partner_name                海尔(字节星选)   
8            actual_train  partner_name          海尔(字节星选拒绝其他通过)   
9            actual_train  partner_name                海尔(字节智选)   
10           actual_train  partner_name                海尔(自营贷中)   
11           actual_train  partner_name                  海尔(蚂蚁)   

metric                partner_count                         \
experiment S1_train_only_unweighted S1_train_only_weighted   
0                               6.0                    6.0   
1                               6.0                    6.0   
2                               1.0                    1.0   
3                               NaN                    NaN   
4                               NaN                    NaN   
5                               1.0                    1.0   
6                               1.0                    1.0   
7                               1.0                    1.0   
8                               1.0                    1.0   
9                               1.0                    1.0   
10                              NaN                    NaN   
11                              NaN                    NaN   

metric                                                 \
experiment S2_train_plus_ensemble_zhongyin_unweighted   
0                                                 7.0   
1                                                 7.0   
2                                                 1.0   
3                                                 NaN   
4                                                 1.0   
5                                                 1.0   
6                                                 1.0   
7                                                 1.0   
8                                                 1.0   
9                                                 1.0   
10                                                NaN   
11                                                NaN   

metric                                               \
experiment S2_train_plus_ensemble_zhongyin_weighted   
0                                               7.0   
1                                               7.0   
2                                               1.0   
3                                               NaN   
4                                               1.0   
5                                               1.0   
6                                               1.0   
7                                               1.0   
8                                               1.0   
9                                               1.0   
10                                              NaN   
11                                              NaN   

metric                                                     \
experiment S3_train_plus_ensemble_non_zhongyin_unweighted   
0                                                     9.0   
1                                                     9.0   
2                                                     1.0   
3                                                     1.0   
4                                                     NaN   
5                                                     1.0   
6                                                     1.0   
7                                                     1.0   
8           

metric     evaluation_set    scope_type            scope_value  \
experiment                                                       
0                   3-oos          mean     partner_macro_mean   
1                   3-oos       overall                    all   
2                   3-oos  partner_name            lexin_type2   
3                   3-oos  partner_name            lexin_type4   
4                   3-oos  partner_name  weipingfubang_highirr   
5                   3-oos  partner_name               海尔(京东贷中)   
6                   3-oos  partner_name               海尔(借呗贷中)   
7                   3-oos  partner_name             海尔(字节星选贷中)   
8                   3-oos  partner_name             海尔(字节智选贷中)   
9                   3-oos  partner_name            海尔(蚂蚁API贷中)   

metric                partner_count                         \
experiment S1_train_only_unweighted S1_train_only_weighted   
0                               8.0                    8.0   
1                               8.0                    8.0   
2                               1.0                    1.0   
3                               1.0                    1.0   
4                               1.0                    1.0   
5                               1.0                    1.0   
6                               1.0                    1.0   
7                               1.0                    1.0   
8                               1.0                    1.0   
9                               1.0                    1.0   

metric                                                 \
experiment S2_train_plus_ensemble_zhongyin_unweighted   
0                                                 8.0   
1                                                 8.0   
2                                                 1.0   
3                                                 1.0   
4                                                 1.0   
5                                                 1.0   
6                                                 1.0   
7                                                 1.0   
8                                                 1.0   
9                                                 1.0   

metric                                               \
experiment S2_train_plus_ensemble_zhongyin_weighted   
0                                               8.0   
1                                               8.0   
2                                               1.0   
3                                               1.0   
4                                               1.0   
5                                               1.0   
6                                               1.0   
7                                               1.0   
8                                               1.0   
9                                               1.0   

metric                                                     \
experiment S3_train_plus_ensemble_non_zhongyin_unweighted   
0                                                     8.0   
1                                                     8.0   
2                                                     1.0   
3                                                     1.0   
4                                                     1.0   
5                                                     1.0   
6                                                     1.0   
7                                                     1.0   
8                                                     1.0   
9                                                     1.0   

metric                                                   \
experiment S3_train_plus_ensemble_non_zhongyin_weighted   
0                                                   8.0   
1                                                   8.0   
2                                                   1.0   
3                                                   1.0   
4                            

metric     evaluation_set    scope_type             scope_value  \
experiment                                                        
0            actual_train          mean      partner_macro_mean   
1            actual_train       overall                     all   
2            actual_train  partner_name             lexin_type5   
3            actual_train  partner_name  weipingfubang_lowerirr   
4            actual_train  partner_name                      中银   
5            actual_train  partner_name                  海尔(借呗)   
6            actual_train  partner_name            海尔(借呗拒绝其他通过)   
7            actual_train  partner_name                海尔(字节星选)   
8            actual_train  partner_name          海尔(字节星选拒绝其他通过)   
9            actual_train  partner_name                海尔(字节智选)   
10           actual_train  partner_name                海尔(自营贷中)   
11           actual_train  partner_name                  海尔(蚂蚁)   
12                  3-oos          mean      partner_macro_mean   
13                  3-oos       overall                     all   
14                  3-oos  partner_name             lexin_type2   
15                  3-oos  partner_name             lexin_type4   
16                  3-oos  partner_name   weipingfubang_highirr   
17                  3-oos  partner_name                海尔(京东贷中)   
18                  3-oos  partner_name                海尔(借呗贷中)   
19                  3-oos  partner_name              海尔(字节星选贷中)   
20                  3-oos  partner_name              海尔(字节智选贷中)   
21                  3-oos  partner_name             海尔(蚂蚁API贷中)   

metric                partner_count                         \
experiment S1_train_only_unweighted S1_train_only_weighted   
0                               6.0                    6.0   
1                               6.0                    6.0   
2                               1.0                    1.0   
3                               NaN                    NaN   
4                               NaN                    NaN   
5                               1.0                    1.0   
6                               1.0                    1.0   
7                               1.0                    1.0   
8                               1.0                    1.0   
9                               1.0                    1.0   
10                              NaN                    NaN   
11                              NaN                    NaN   
12                              8.0                    8.0   
13                              8.0                    8.0   
14                              1.0                    1.0   
15                              1.0                    1.0   
16                              1.0                    1.0   
17                              1.0                    1.0   
18                              1.0                    1.0   
19                              1.0                    1.0   
20                              1.0                    1.0   
21                              1.0                    1.0   

metric                                                 \
experiment S2_train_plus_ensemble_zhongyin_unweighted   
0                                                 7.0   
1                                                 7.0   
2                                                 1.0   
3                                                 NaN   
4                                                 1.0   
5                                                 1.0   
6                                                 1.0   
7                                                 1.0   
8                                                 1.0   
9                                                 1.0   
10                                                NaN   
11                                                NaN   
12                                                8.0   
13                                             

,scenario,scope_type,scope_value,experiment,partner_count,n,bad_count,bad_rate,coverage,model_auc,model_ks,model_score_direction,base_auc,base_ks,base_score_direction,delta_auc,delta_ks,evaluation_set
0,S1_train_only,overall,all,S1_train_only_unweighted,6,735017,18755,0.025516,1.000000,0.654063,0.216468,higher_is_bad,0.637756,0.194369,higher_is_bad,0.016307,0.022099,actual_train
1,S1_train_only,overall,all,S1_train_only_weighted,6,735017,18755,0.025516,1.000000,0.631754,0.186643,higher_is_bad,0.637756,0.194369,higher_is_bad,-0.006003,-0.007725,actual_train
2,S2_train_plus_ensemble_zhongyin,overall,all,S2_train_plus_ensemble_zhongyin_unweighted,7,1084406,52646,0.048548,0.990310,0.698049,0.290254,higher_is_bad,0.603620,0.148010,higher_is_bad,0.094429,0.142244,actual_train
3,S2_train_plus_ensemble_zhongyin,overall,all,S2_train_plus_ensemble_zhongyin_weighted,7,1084406,52646,0.048548,0.990310,0.607830,0.153845,higher_is_bad,0.603620,0.148010,higher_is_bad,0.004209,0.005835,actual_train
4,S3_train_plus_ensemble_non_zhongyin,overall,all,S3_train_plus_ensemble_non_zhongyin_unweighted,9,1195645,62094,0.051933,1.000000,0.695360,0.290116,higher_is_bad,0.661293,0.234421,higher_is_bad,0.034066,0.055696,actual_train
5,S3_train_plus_ensemble_non_zhongyin,overall,all,S3_train_plus_ensemble_non_zhongyin_weighted,9,1195645,62094,0.051933,1.000000,0.629863,0.184640,higher_is_bad,0.661293,0.234421,higher_is_bad,-0.031430,-0.049780,actual_train
6,S4_train_plus_all_ensemble,overall,all,S4_train_plus_all_ensemble_unweighted,10,1545034,95985,0.062125,0.993179,0.690512,0.276586,higher_is_bad,0.631914,0.190252,higher_is_bad,0.058598,0.086334,actual_train
7,S4_train_plus_all_ensemble,overall,all,S4_train_plus_all_ensemble_weighted,10,1545034,95985,0.062125,0.993179,0.610417,0.165392,higher_is_bad,0.631914,0.190252,higher_is_bad,-0.021497,-0.024860,actual_train
8,S1_train_only,mean,partner_macro_mean,S1_train_only_unweighted,6,735017,18755,0.025516,1.000000,0.656897,0.230328,partner_macro_mean,0.636757,0.200408,partner_macro_mean,0.020140,0.029920,actual_train
9,S1_train_only,mean,partner_macro_mean,S1_train_only_weighted,6,735017,18755,0.025516,1.000000,0.703682,0.305485,partner_macro_mean,0.636757,0.200408,partner_macro_mean,0.066925,0.105077,actual_train


metric     evaluation_set scope_type         scope_value  \
experiment                                                 
0            actual_train       mean  partner_macro_mean   
1            actual_train    overall                 all   
2                   3-oos       mean  partner_macro_mean   
3                   3-oos    overall                 all   

metric                partner_count                         \
experiment S1_train_only_unweighted S1_train_only_weighted   
0                               6.0                    6.0   
1                               6.0                    6.0   
2                               8.0                    8.0   
3                               8.0                    8.0   

metric                                                 \
experiment S2_train_plus_ensemble_zhongyin_unweighted   
0                                                 7.0   
1                                                 7.0   
2                                                 8.0   
3                                                 8.0   

metric                                               \
experiment S2_train_plus_ensemble_zhongyin_weighted   
0                                               7.0   
1                                               7.0   
2                                               8.0   
3                                               8.0   

metric                                                     \
experiment S3_train_plus_ensemble_non_zhongyin_unweighted   
0                                                     9.0   
1                                                     9.0   
2                                                     8.0   
3                                                     8.0   

metric                                                   \
experiment S3_train_plus_ensemble_non_zhongyin_weighted   
0                                                   9.0   
1                                                   9.0   
2                                                   8.0   
3                                                   8.0   

metric                                            \
experiment S4_train_plus_all_ensemble_unweighted   
0                                           10.0   
1                                           10.0   
2                                            8.0   
3                                            8.0   

metric                                                                n  \
experiment S4_train_plus_all_ensemble_weighted S1_train_only_unweighted   
0                                         10.0                 735017.0   
1                                         10.0                 735017.0   
2                                          8.0                1626010.0   
3                                          8.0                1626010.0   

metric                                                                        \
experiment S1_train_only_weighted S2_train_plus_ensemble_zhongyin_unweighted   
0                        735017.0                                  1084406.0   
1                        735017.0                                  1084406.0   
2                       1626010.0                                  1626010.0   
3                       1626010.0                                  1626010.0   

metric                                               \
experiment S2_train_plus_ensemble_zhongyin_weighted   
0                                         1084406.0   
1                                         1084406.0   
2                                         1626010.0   
3                                         1626010.0   

metric                                                     \
experiment S3_train_plus_ensemble_non_zhongyin_unweighted   
0                                               1195645.0   
1                                               1195645.0   
2                                               16

In [81]:
result_wide.to_excel('2-result_wide.xlsx')

In [85]:
prediction_df.head()

,test_id,flag,partner_name,partner_code,mid_sample_type,ym_proba_a077_3,S1_train_only_unweighted,S1_train_only_weighted,S2_train_plus_ensemble_zhongyin_unweighted,S2_train_plus_ensemble_zhongyin_weighted,S3_train_plus_ensemble_non_zhongyin_unweighted,S3_train_plus_ensemble_non_zhongyin_weighted,S4_train_plus_all_ensemble_unweighted,S4_train_plus_all_ensemble_weighted
861159,4491235,0.0,lexin_type2,c_19,3-oos,0.015791,0.017216,0.018238,0.019398,0.015298,0.049208,0.014765,0.045713,0.304888
861160,4491238,0.0,lexin_type2,c_19,3-oos,0.022935,0.020459,0.023324,0.032490,0.029692,0.062692,0.026999,0.063407,0.308840
861161,4491240,0.0,lexin_type2,c_19,3-oos,0.030106,0.029071,0.038350,0.042917,0.041099,0.046265,0.037241,0.057282,0.312471
861162,4491242,0.0,lexin_type2,c_19,3-oos,0.050666,0.031867,0.046754,0.064746,0.052465,0.065718,0.046410,0.080495,0.314629
861163,4491248,0.0,lexin_type2,c_19,3-oos,0.040962,0.034504,0.066152,0.058848,0.077485,0.053201,0.060547,0.066731,0.321247


# 分析每种实验下的partner_name=中银的概率评分分布情况


In [94]:
ANALYSIS_SAMPLE_TYPES = [
    '1-train',
    '2-ensemble'
]

# 保留该变量名，兼容后续统计代码
ANALYSIS_SAMPLE_TYPE = (
    '1-train + 2-ensemble + 3-oos'
)

ZHONGYIN_NAME = '中银'

analysis_df = prediction_df.loc[
    prediction_df[SAMPLE_COL].isin(
        ANALYSIS_SAMPLE_TYPES
    )
].copy()

if analysis_df.empty:
    raise ValueError(
        '1-train、2-ensemble、3-oos '
        '均没有可分析样本'
    )

analysis_df['partner_group'] = np.where(
    analysis_df[PARTNER_COL].eq(
        ZHONGYIN_NAME
    ),
    '中银',
    '其他渠道'
)

required_score_cols = set(
    experiment_order
)
missing_score_cols = (
    required_score_cols
    - set(analysis_df.columns)
)

if missing_score_cols:
    raise KeyError(
        'prediction_df 缺少实验预测列: {}'.format(
            sorted(missing_score_cols)
        )
    )

# 检查每个实验在各样本类型、渠道组上的预测覆盖情况
coverage_rows = []

for experiment_name in experiment_order:
    current_score = pd.to_numeric(
        analysis_df[experiment_name],
        errors='coerce'
    )

    current_valid = (
        current_score.notna()
        & np.isfinite(current_score)
    )

    current_df = analysis_df[
        [
            SAMPLE_COL,
            PARTNER_COL,
            'partner_group'
        ]
    ].copy()

    current_df['score_valid'] = (
        current_valid
    )

    current_coverage = (
        current_df
        .groupby(
            [
                SAMPLE_COL,
                'partner_group'
            ],
            dropna=False
        )
        .agg(
            sample_n=(
                'score_valid',
                'size'
            ),
            score_n=(
                'score_valid',
                'sum'
            )
        )
        .reset_index()
    )

    current_coverage[
        'score_coverage'
    ] = (
        current_coverage['score_n']
        / current_coverage['sample_n']
    )

    current_coverage.insert(
        0,
        'experiment',
        experiment_name
    )

    coverage_rows.append(
        current_coverage
    )

score_coverage_by_sample_type = (
    pd.concat(
        coverage_rows,
        ignore_index=True
    )
)

display(
    score_coverage_by_sample_type
)

,experiment,mid_sample_type,partner_group,sample_n,score_n,score_coverage
0,S1_train_only_unweighted,1-train,其他渠道,735017,735017,1.0
1,S1_train_only_unweighted,2-ensemble,中银,360000,0,0.0
2,S1_train_only_unweighted,2-ensemble,其他渠道,460628,0,0.0
3,S1_train_only_weighted,1-train,其他渠道,735017,735017,1.0
4,S1_train_only_weighted,2-ensemble,中银,360000,0,0.0
5,S1_train_only_weighted,2-ensemble,其他渠道,460628,0,0.0
6,S2_train_plus_ensemble_zhongyin_unweighted,1-train,其他渠道,735017,735017,1.0
7,S2_train_plus_ensemble_zhongyin_unweighted,2-ensemble,中银,360000,360000,1.0
8,S2_train_plus_ensemble_zhongyin_unweighted,2-ensemble,其他渠道,460628,0,0.0
9,S2_train_plus_ensemble_zhongyin_weighted,1-train,其他渠道,735017,735017,1.0


In [95]:

SCORE_QUANTILES = [
    0.01,
    0.05,
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99
]

def score_distribution_stats(
    score,
    quantiles=SCORE_QUANTILES
):
    score = pd.to_numeric(
        pd.Series(score),
        errors='coerce'
    )
    score = score.loc[
        score.notna()
        & np.isfinite(score)
    ]

    result = {
        'n': int(len(score)),
        'mean': (
            float(score.mean())
            if len(score) else np.nan
        ),
        'std': (
            float(score.std())
            if len(score) > 1 else np.nan
        ),
        'min': (
            float(score.min())
            if len(score) else np.nan
        ),
        'max': (
            float(score.max())
            if len(score) else np.nan
        )
    }

    if len(score):
        quantile_result = score.quantile(
            quantiles
        )

        for quantile, value in (
            quantile_result.items()
        ):
            result[
                'p{:02d}'.format(
                    int(quantile * 100)
                )
            ] = float(value)
    else:
        for quantile in quantiles:
            result[
                'p{:02d}'.format(
                    int(quantile * 100)
                )
            ] = np.nan

    return result

In [96]:

distribution_rows = []
comparison_rows = []
partner_mean_rows = []

for experiment_name in experiment_order:
    score_df = analysis_df[
        [
            SAMPLE_COL,
            PARTNER_COL,
            'partner_group',
            experiment_name
        ]
    ].copy()

    score_df['score'] = pd.to_numeric(
        score_df[experiment_name],
        errors='coerce'
    )

    score_df = score_df.loc[
        score_df['score'].notna()
        & np.isfinite(score_df['score'])
    ]

    # 中银样本
    zhongyin_df = score_df.loc[
        score_df['partner_group'].eq(
            '中银'
        )
    ]

    # 其他全部样本：
    # 包括 1-train、2-ensemble非中银、3-oos
    other_df = score_df.loc[
        score_df['partner_group'].eq(
            '其他渠道'
        )
    ]

    zhongyin_stats = (
        score_distribution_stats(
            zhongyin_df['score']
        )
    )
    zhongyin_stats.update({
        'experiment': experiment_name,
        'analysis_scope':
            ANALYSIS_SAMPLE_TYPE,
        'partner_group': '中银'
    })
    distribution_rows.append(
        zhongyin_stats
    )

    other_stats = (
        score_distribution_stats(
            other_df['score']
        )
    )
    other_stats.update({
        'experiment': experiment_name,
        'analysis_scope':
            ANALYSIS_SAMPLE_TYPE,
        'partner_group': '其他渠道'
    })
    distribution_rows.append(
        other_stats
    )

    # 每个 partner_name 的平均概率
    current_partner_mean = (
        score_df
        .groupby(
            PARTNER_COL,
            dropna=False
        )['score']
        .agg(
            sample_n='size',
            score_mean='mean',
            score_std='std',
            score_median='median',
            score_min='min',
            score_max='max'
        )
        .reset_index()
    )

    current_partner_mean.insert(
        0,
        'experiment',
        experiment_name
    )
    partner_mean_rows.append(
        current_partner_mean
    )

    zhongyin_mean = (
        float(
            zhongyin_df['score'].mean()
        )
        if len(zhongyin_df)
        else np.nan
    )

    # 其他渠道全部样本合并后的平均值
    other_pooled_mean = (
        float(
            other_df['score'].mean()
        )
        if len(other_df)
        else np.nan
    )

    # 每个其他 partner_name 先算均值，
    # 再对渠道均值做不加权平均
    other_partner_means = (
        current_partner_mean.loc[
            ~current_partner_mean[
                PARTNER_COL
            ].eq(ZHONGYIN_NAME),
            'score_mean'
        ]
    )

    other_partner_macro_mean = (
        float(
            other_partner_means.mean()
        )
        if len(other_partner_means)
        else np.nan
    )

    comparison_rows.append({
        'experiment':
            experiment_name,
        'analysis_scope':
            ANALYSIS_SAMPLE_TYPE,

        'zhongyin_n':
            int(len(zhongyin_df)),
        'other_n':
            int(len(other_df)),

        'other_train_n': int(
            other_df[SAMPLE_COL]
            .eq('1-train')
            .sum()
        ),
        'other_ensemble_n': int(
            other_df[SAMPLE_COL]
            .eq('2-ensemble')
            .sum()
        ),
        'other_oos_n': int(
            other_df[SAMPLE_COL]
            .eq('3-oos')
            .sum()
        ),

        'zhongyin_mean':
            zhongyin_mean,
        'other_pooled_mean':
            other_pooled_mean,
        'other_partner_macro_mean':
            other_partner_macro_mean,

        'zhongyin_vs_other_pooled_delta': (
            zhongyin_mean
            - other_pooled_mean
        ),
        'zhongyin_vs_other_macro_delta': (
            zhongyin_mean
            - other_partner_macro_mean
        ),

        'zhongyin_vs_other_pooled_ratio': (
            zhongyin_mean
            / other_pooled_mean
            if (
                np.isfinite(
                    other_pooled_mean
                )
                and other_pooled_mean != 0
            )
            else np.nan
        ),

        'zhongyin_vs_other_macro_ratio': (
            zhongyin_mean
            / other_partner_macro_mean
            if (
                np.isfinite(
                    other_partner_macro_mean
                )
                and other_partner_macro_mean != 0
            )
            else np.nan
        )
    })

score_distribution_long = (
    pd.DataFrame(
        distribution_rows
    )
)

score_comparison = (
    pd.DataFrame(
        comparison_rows
    )
    .set_index('experiment')
    .reindex(experiment_order)
    .reset_index()
)

partner_score_mean_long = (
    pd.concat(
        partner_mean_rows,
        ignore_index=True
    )
)

display(score_comparison)
display(score_distribution_long)
display(partner_score_mean_long)

,experiment,analysis_scope,zhongyin_n,other_n,other_train_n,other_ensemble_n,other_oos_n,zhongyin_mean,other_pooled_mean,other_partner_macro_mean,zhongyin_vs_other_pooled_delta,zhongyin_vs_other_macro_delta,zhongyin_vs_other_pooled_ratio,zhongyin_vs_other_macro_ratio
0,S1_train_only_unweighted,1-train + 2-ensemble + 3-oos,0,735017,735017,0,0,NaN,0.025454,0.027146,NaN,NaN,NaN,NaN
1,S1_train_only_weighted,1-train + 2-ensemble + 3-oos,0,735017,735017,0,0,NaN,0.039955,0.039815,NaN,NaN,NaN,NaN
2,S2_train_plus_ensemble_zhongyin_unweighted,1-train + 2-ensemble + 3-oos,360000,735017,735017,0,0,0.065168,0.041275,0.043401,0.023892,0.021766,1.578851,1.501511
3,S2_train_plus_ensemble_zhongyin_weighted,1-train + 2-ensemble + 3-oos,360000,735017,735017,0,0,0.037644,0.039833,0.040211,-0.002190,-0.002567,0.945026,0.936155
4,S3_train_plus_ensemble_non_zhongyin_unweighted,1-train + 2-ensemble + 3-oos,0,1195645,735017,460628,0,NaN,0.051853,0.052657,NaN,NaN,NaN,NaN
5,S3_train_plus_ensemble_non_zhongyin_weighted,1-train + 2-ensemble + 3-oos,0,1195645,735017,460628,0,NaN,0.040066,0.039860,NaN,NaN,NaN,NaN
6,S4_train_plus_all_ensemble_unweighted,1-train + 2-ensemble + 3-oos,360000,1195645,735017,460628,0,0.065408,0.061584,0.062576,0.003823,0.002831,1.062084,1.045243
7,S4_train_plus_all_ensemble_weighted,1-train + 2-ensemble + 3-oos,360000,1195645,735017,460628,0,0.311641,0.313528,0.313541,-0.001887,-0.001901,0.993980,0.993937


,n,mean,std,min,max,p01,p05,p10,p25,p50,p75,p90,p95,p99,experiment,analysis_scope,partner_group
0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,S1_train_only_unweighted,1-train + 2-ensemble + 3-oos,中银
1,735017,0.025454,0.013078,0.003673,0.187572,0.007912,0.010430,0.012550,0.017432,0.022786,0.029951,0.040624,0.050041,0.073212,S1_train_only_unweighted,1-train + 2-ensemble + 3-oos,其他渠道
2,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,S1_train_only_weighted,1-train + 2-ensemble + 3-oos,中银
3,735017,0.039955,0.020414,0.006612,0.398588,0.013396,0.016769,0.019625,0.026832,0.035670,0.048031,0.063266,0.077079,0.112588,S1_train_only_weighted,1-train + 2-ensemble + 3-oos,其他渠道
4,360000,0.065168,0.037096,0.007635,0.676647,0.016703,0.022997,0.027466,0.038504,0.056480,0.085018,0.110115,0.130455,0.185133,S2_train_plus_ensemble_zhongyin_unweighted,1-train + 2-ensemble + 3-oos,中银
5,735017,0.041275,0.018587,0.006078,0.282487,0.015147,0.019427,0.022143,0.028061,0.037352,0.050033,0.065306,0.076628,0.103021,S2_train_plus_ensemble_zhongyin_unweighted,1-train + 2-ensemble + 3-oos,其他渠道
6,360000,0.037644,0.018769,0.008081,0.324042,0.014078,0.016733,0.018770,0.024357,0.034037,0.045055,0.059885,0.071188,0.106317,S2_train_plus_ensemble_zhongyin_weighted,1-train + 2-ensemble + 3-oos,中银
7,735017,0.039833,0.021041,0.006102,0.568541,0.013860,0.017039,0.019498,0.025876,0.035032,0.048154,0.064623,0.078356,0.116263,S2_train_plus_ensemble_zhongyin_weighted,1-train + 2-ensemble + 3-oos,其他渠道
8,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,S3_train_plus_ensemble_non_zhongyin_unweighted,1-train + 2-ensemble + 3-oos,中银
9,1195645,0.051853,0.034455,0.006481,0.334851,0.011359,0.015132,0.018690,0.026534,0.041253,0.068711,0.100171,0.120685,0.165599,S3_train_plus_ensemble_non_zhongyin_unweighted,1-train + 2-ensemble + 3-oos,其他渠道


,experiment,partner_name,sample_n,score_mean,score_std,score_median,score_min,score_max
0,S1_train_only_unweighted,lexin_type5,34036,0.021268,0.007541,0.020621,0.005639,0.106800
1,S1_train_only_unweighted,海尔(借呗),452145,0.024489,0.011273,0.022581,0.003673,0.176095
2,S1_train_only_unweighted,海尔(借呗拒绝其他通过),95095,0.034704,0.019358,0.029874,0.004845,0.187572
3,S1_train_only_unweighted,海尔(字节星选),133260,0.022866,0.011002,0.020967,0.004781,0.174362
4,S1_train_only_unweighted,海尔(字节星选拒绝其他通过),5034,0.034004,0.019836,0.028636,0.006421,0.175058
5,S1_train_only_unweighted,海尔(字节智选),15447,0.025549,0.011800,0.023178,0.004931,0.148096
6,S1_train_only_weighted,lexin_type5,34036,0.035932,0.013424,0.033656,0.010529,0.252901
7,S1_train_only_weighted,海尔(借呗),452145,0.041025,0.020164,0.036833,0.007086,0.347300
8,S1_train_only_weighted,海尔(借呗拒绝其他通过),95095,0.042503,0.025153,0.036504,0.006637,0.398588
9,S1_train_only_weighted,海尔(字节星选),133260,0.035197,0.017140,0.032086,0.006612,0.301199


In [97]:
score_comparison.to_excel('1-score_comparison.xlsx')

In [93]:
partner_score_mean_long.to_excel('1-partner_score_mean_long.xlsx')

In [74]:
# ============================================================
# 1. 在八组模型训练循环中，同时保存 actual_train 和 3-oos 预测
#    用下面代码替换原来的 train_pred / prediction_df 赋值部分
# ============================================================

prediction_df[experiment_name] = np.nan

# 实际训练集预测
actual_train_pred = experiment_model.predict_proba(
    scenario_df[feature_cols]
)[:, 1]

if not np.isfinite(actual_train_pred).all():
    raise ValueError(
        '{} 的 actual_train 预测包含 NaN/inf'.format(
            experiment_name
        )
    )

prediction_df.loc[
    scenario_df.index,
    experiment_name
] = actual_train_pred

# 3-oos 预测
oos_feature_df = model_df.loc[
    oos_mask,
    feature_cols
]

oos_pred = experiment_model.predict_proba(
    oos_feature_df
)[:, 1]

if not np.isfinite(oos_pred).all():
    raise ValueError(
        '{} 的 3-oos 预测包含 NaN/inf'.format(
            experiment_name
        )
    )

prediction_df.loc[
    oos_feature_df.index,
    experiment_name
] = oos_pred

del (
    experiment_model,
    actual_train_pred,
    oos_pred,
    oos_feature_df,
    sample_weight
)
gc.collect()

# ============================================================
# 2. 通用评估函数：同时支持 actual_train 和 3-oos
# ============================================================

def append_evaluation_rows(
    result_rows,
    eval_df,
    evaluation_set,
    experiment_name,
    scenario_name
):
    if eval_df.empty:
        raise ValueError(
            '{} / {} 的评估数据为空'.format(
                experiment_name,
                evaluation_set
            )
        )

    # overall
    overall_row = compare_model_with_base(
        eval_df=eval_df,
        pred_col=experiment_name,
        scenario_name=scenario_name,
        scope_type='overall',
        scope_value='all'
    )
    overall_row['evaluation_set'] = evaluation_set
    result_rows.append(overall_row)

    # partner_name
    partner_rows = []

    for partner_name, partner_df in eval_df.groupby(
        PARTNER_COL,
        dropna=False,
        sort=True
    ):
        partner_row = compare_model_with_base(
            eval_df=partner_df,
            pred_col=experiment_name,
            scenario_name=scenario_name,
            scope_type='partner_name',
            scope_value=str(partner_name)
        )
        partner_row['evaluation_set'] = evaluation_set

        partner_rows.append(partner_row)
        result_rows.append(partner_row)

    # partner macro mean
    mean_row = build_partner_macro_mean(
        partner_rows=partner_rows,
        scenario_name=scenario_name,
        experiment_name=experiment_name,
        overall_row=overall_row
    )
    mean_row['evaluation_set'] = evaluation_set
    result_rows.append(mean_row)

# ============================================================
# 3. 分别计算八组实验的 actual_train 和 3-oos 指标
# ============================================================

result_rows = []

for registry_row in experiment_registry.itertuples(
    index=False
):
    experiment_name = registry_row.experiment
    scenario_name = registry_row.scenario

    # 当前实验的实际训练范围
    actual_train_mask = TRAIN_SCENARIOS[
        scenario_name
    ].reindex(
        prediction_df.index,
        fill_value=False
    )

    actual_train_df = prediction_df.loc[
        actual_train_mask
    ].copy()

    append_evaluation_rows(
        result_rows=result_rows,
        eval_df=actual_train_df,
        evaluation_set='actual_train',
        experiment_name=experiment_name,
        scenario_name=scenario_name
    )

    # 所有实验统一使用同一批 3-oos
    experiment_oos_df = prediction_df.loc[
        prediction_df[SAMPLE_COL].eq('3-oos')
    ].copy()

    append_evaluation_rows(
        result_rows=result_rows,
        eval_df=experiment_oos_df,
        evaluation_set='3-oos',
        experiment_name=experiment_name,
        scenario_name=scenario_name
    )

# ============================================================
# 4. result_long
# ============================================================

result_long = pd.DataFrame(result_rows)

evaluation_order = {
    'actual_train': 0,
    '3-oos': 1
}
scope_order = {
    'overall': 0,
    'mean': 1,
    'partner_name': 2
}
experiment_order_map = {
    experiment_name: order
    for order, experiment_name
    in enumerate(experiment_order)
}

result_long['_evaluation_order'] = (
    result_long['evaluation_set']
    .map(evaluation_order)
)
result_long['_scope_order'] = (
    result_long['scope_type']
    .map(scope_order)
)
result_long['_experiment_order'] = (
    result_long['experiment']
    .map(experiment_order_map)
)

result_long = (
    result_long
    .sort_values([
        '_evaluation_order',
        '_scope_order',
        'scope_value',
        '_experiment_order'
    ])
    .drop(columns=[
        '_evaluation_order',
        '_scope_order',
        '_experiment_order'
    ])
    .reset_index(drop=True)
)

result_long_actual_train = (
    result_long.loc[
        result_long['evaluation_set'].eq(
            'actual_train'
        )
    ]
    .reset_index(drop=True)
)

result_long_oos = (
    result_long.loc[
        result_long['evaluation_set'].eq(
            '3-oos'
        )
    ]
    .reset_index(drop=True)
)

# ============================================================
# 5. result_wide 生成函数
# ============================================================

WIDE_METRIC_COLS = [
    'partner_count',
    'n',
    'bad_count',
    'bad_rate',
    'coverage',
    'model_auc',
    'base_auc',
    'delta_auc',
    'model_ks',
    'base_ks',
    'delta_ks'
]

def build_result_wide(
    result_long_df,
    experiment_order
):
    if result_long_df.empty:
        return pd.DataFrame()

    result_wide_df = result_long_df.pivot(
        index=[
            'evaluation_set',
            'scope_type',
            'scope_value'
        ],
        columns='experiment',
        values=WIDE_METRIC_COLS
    )

    expected_columns = pd.MultiIndex.from_product(
        [
            WIDE_METRIC_COLS,
            experiment_order
        ],
        names=[
            'metric',
            'experiment'
        ]
    )

    result_wide_df = (
        result_wide_df
        .reindex(columns=expected_columns)
        .reset_index()
    )

    return result_wide_df

# ============================================================
# 6. 分别生成 actual_train / OOS result_wide，再拼接
# ============================================================

result_wide_actual_train = build_result_wide(
    result_long_df=result_long_actual_train,
    experiment_order=experiment_order
)

result_wide_oos = build_result_wide(
    result_long_df=result_long_oos,
    experiment_order=experiment_order
)

result_wide = pd.concat(
    [
        result_wide_actual_train,
        result_wide_oos
    ],
    axis=0,
    ignore_index=True
)

# ============================================================
# 7. 只保留 overall + mean 的汇总结果
# ============================================================

result_long_summary = (
    result_long.loc[
        result_long['scope_type'].isin([
            'overall',
            'mean'
        ])
    ]
    .reset_index(drop=True)
)

result_wide_summary = (
    result_wide.loc[
        result_wide['scope_type'].isin([
            'overall',
            'mean'
        ])
    ]
    .reset_index(drop=True)
)

display(result_long_actual_train)
display(result_long_oos)

display(result_wide_actual_train)
display(result_wide_oos)

# actual_train 与 3-oos 拼接后的最终宽表
display(result_wide)

# 仅 overall + mean
display(result_long_summary)
display(result_wide_summary)


NameError: name 'experiment_model' is not defined

# 审计摘要

In [ ]:
audit_summary = pd.DataFrame([
    {'item': 'feature_source', 'value': 'ym_hidden80.pkl + ym_v4.pkl'},
    {'item': 'hidden_feature_count', 'value': len(hidden_feature_cols)},
    {'item': 'v4_feature_count', 'value': len(v4_feature_cols)},
    {'item': 'total_feature_count', 'value': len(feature_cols)},
    {'item': 'merged_row_count', 'value': len(model_df)},
    {'item': 'weight_train_scope', 'value': '1-train'},
    {'item': 'weight_validation_scope', 'value': '2-ensemble & partner_name != 中银'},
    {'item': 'oos_used_for_tuning', 'value': False},
    {'item': 'selected_base_cnt', 'value': best_weight_params['base_cnt']},
    {'item': 'selected_base_rate', 'value': best_weight_params['base_rate']},
    {'item': 'experiment_count', 'value': len(experiment_registry)},
    {'item': 'evaluation_scope', 'value': '按 model_df.mid_sample_type 分组评估全部有标签样本'},
    {'item': 'official_oos_scope', 'value': '3-oos'},
    {'item': 'mean_definition', 'value': 'partner_name 指标不加权算术平均（macro mean）'},
    {'item': 'base_score', 'value': BASE_COL},
    {'item': 'score_transform', 'value': '原始分数；未自动执行 1-score'}
])
display(audit_summary)